# 手順5b：最終候補（B_1・C_1）を MCMC で推定

CLAUDE.md §5・§7。手順5 で決まった最終候補 **B_1**（構造 B：温度で横にも縦にもずらす。厚み H を残す）と、比較用の **C_1**（構造なし。手順3b・3d の R3）を、MCMC（自作の適応型ランダムウォーク・メトロポリス法。手順2・3c と同じ）でも推定し、最尤推定と比べる。B_1 は、温度のずらし量 E_h・E_v も、ハイパーパラメータと**同時にサンプリング**する（ユーザー決定、10-14）。データは、手順3d の 2 つの版（5data 版・Alldata 版）。

- **何を確かめるか**：(1) E_h・E_v の事後分布（手順5 のプロファイル尤度の結果と合うか。斜めの帯〔差 E_v − E_h が決まる〕になるか）。(2) ハイパーパラメータが、探索範囲の端に張り付いていないか。(3) ハイパーパラメータと E の不確かさを予測に含めたとき、予測区間の広さ・被覆率・NLPD が、最尤推定とどれだけ違うか。(4) 設定温度の効果（190/230 ℃ の粘度比）の不確かさと、実測との関係。

## モデルの構成（手順5 と同じ）

| 項目 | 内容 |
|---|---|
| 学習データ | 外れ値なし。問題データ一覧の「除外」（P01・P03・P08）を除く。1 行＝1 ショット。5data 版：5data（PP 460 行・混合 459 行）／Alldata 版：Alldata（PP 1234 行・混合 560 行） |
| 目的変数 | ln η（η：見かけの粘度 [Pa·s]）。学習データの平均 y0 を引く（固定の定数。自動標準化はしない） |
| 入力：C_1 | 条件平均の ln γ̇、ln H、Tset［℃］（3 列。学習データの平均・標準偏差で標準化） |
| 入力：B_1 | u = 条件平均の ln γ̇ ＋ ln a(Tset; E_h)、w = ln Fo = −(ln γ̇ ＋ 3 ln H)（2 列）。出力から ln a(Tset; E_v) を引く。ln a(T; E) = (E/R)(1/T − 1/Tref)、Tref = 483.15 K、R = 8.314 |
| 平均関数 | 明示的なものはない（ガウス過程の平均はゼロ）。定数 y0、B_1 の縦シフト ln a(Tset; E_v)、カーネルの線形項が、平均の役目をする |
| カーネル | C1·Matern(ν = 5/2、入力ごとの長さスケール〔ARD〕) ＋ C2·DotProduct(σ0) ＋ ノイズ（WhiteKernel、分散 σn²） |
| 事前分布（カーネル） | 手順3c と同じ。探索範囲の中で、対数スケールの一様分布（C1・C2：1e-3〜1e3、長さスケール：1e-2〜1e2、σ0：1e-5〜1e3、σn²：1e-6〜10）。範囲の外は確率 0 |
| 事前分布（E） | 範囲の中で一様分布（`E_PRIOR`。E_h：−40〜40、E_v：−20〜40 kJ/mol。手順5 のプロファイル尤度の格子と同じ範囲） |
| 関数の事前分布 | f ~ ガウス過程（平均 0、共分散 上のカーネル）。ノイズ ε ~ N(0, σn²) |

**条件ごとにまとめた尤度（厳密）**：入力が条件平均なので、同じ条件のショットは入力が同じ。ショットごとの尤度は、条件ごとの平均・ショット数・条件内の平方和を使って、厳密に「条件内のばらつきの項」と「条件平均の GP の項」に分けられる（`step5b_core.py`）。条件の数（93〜129）の大きさの行列だけで済むので、MCMC の 1 回の評価が約 1 ミリ秒になる（Alldata 版でも同じ）。ノートの最初の節で、scikit-learn のショットごとの GPR と、対数周辺尤度・予測が一致することを確かめる。

**予測**：事後サンプルを等間隔に `N_PRED_DRAWS` 個選び、各サンプルでの予測分布（ノイズ込みの正規分布）を等しい重みで混ぜた混合分布。P3_2 を使うモデル（H の扱い 4・14・24）を指定したときは、実測の P3_2（条件平均）で予測する。

### 自作の適応型ランダムウォーク・メトロポリス法（手順3c と同じ）
1. 今いる点 φ（カーネルの対数パラメータと E）の近くに、正規分布 N(φ, λ·Σ) から候補を出す。
2. 候補の対数事後密度（範囲内なら対数尤度、範囲外なら −∞）を計算し、確率 min(1, exp(候補 − 現在)) で移動する。
3. **調整期間**：前半 75% では、それまでの足跡から Σ（パラメータどうしの相関を含む）を学ぶ。調整期間の間ずっと、採択率が 0.234 に近づくよう λ を合わせる。調整期間のサンプルは捨てる。
4. **本番**：Σ と λ を固定して歩く。R-hat が 1.01 を超えるか、有効サンプル数（ESS）が 400 に届かないときは、本番の回数を増やして、サンプリングし直す（最大 `N_SAMP_MAX` 回）。
5. 4 本のチェーンを別々の初期値（最尤推定値に、ゆらぎを足したもの）から回し、R-hat と ESS で収束を確かめる。

## 使い方（このノートを一人で実行して、結果を判断するために）

1. メニューの「すべて実行」で、上から順に実行する（約 30 分）。カーネルは、このプロジェクトの `.venv` を選ぶ。
2. 設定を変えたいときは、最初の「設定エリア」のセルだけを直して、最初から実行し直す（高解像度のグラフの設定は、下の 7）。
3. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」。基準は、判定の文の中に書いてある。★ が出ても、エラーではない。説明書の「判定の読み方」で、★ の意味を確かめる。
4. 図と表は `results/step5b_mcmc/` に保存される。図の見方は、説明書 `results/学習結果説明_手順5b.pdf` にある。
5. データを取り直したとき、問題のあるデータを入れ替えたときは、`問題データ一覧.md` の手順に従う。このノートは、全条件のファイルを読み、`02_機械学習データ/問題データ一覧.csv` の「除外」を除く。
6. 用語や図の見方で分からないところは、説明書の用語集で調べる。記号（Tff、Tset、R3、E_h など）の意味は、`results/学習結果説明_記号表.pdf` に一覧がある。
7. **高解像度のグラフ**（報告書・発表資料用。GPR6月ノートのセル9-2 と同じ見せ方）は、「高解像度のグラフ」の節のセルで作る。**このセルだけは、セルの先頭にも設定エリアがあり、そこだけを直して、このセルだけを実行し直せる**（学習はやり直さない。全体を一度実行した後で）。保存先は `results/step5b_mcmc/highres/`（Git には入れない）。
8. サンプリングには時間がかかる（約 30 分）。`USE_SAVED_SAMPLES = True` のときは、保存済みの事後サンプル（`results/step5b_mcmc/posterior_*.csv`）があれば読み込み、サンプリングを省く。データを取り直したときは `False` にする（`rerun_all.py` は、実行の前に保存済みのサンプルを消す）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
SEED = 42                     # 乱数シード
NOISE_LB = 1e-6               # ノイズ分散の下限（旧ノートは 1e-3）
N_RESTARTS = 5                # GPR の最適化のやり直し回数（増やすと遅いが、安定する）
USE_EXCLUSIONS = True         # True：問題データ一覧の「除外」を使う／False：全条件を使う（取り直した後の確認用）
REPO = ".."                   # リポジトリ直下（このノートは 01_機械学習コードフォルダ にある）
FIG_DPI = 200                 # 保存する PNG の解像度
import numpy as np             # 範囲の指定に使う
MCMC_MODELS = ["B_1", "C_1"]  # MCMC で推定するモデル（「構造_H の扱い」。手順5 の 15 モデルのどれでもよい。H の扱い：1＝H を残す、2＝m＋c(Fo)、4＝P3_2、14、24）
SETS5B = {"5data": ["S_V", "S_T"],          # 版ごとの学習データ。5data 版：S_V（5data で学習。3data を検証）・S_T（190・230 ℃ で学習。210 ℃ を検証）
          "Alldata": ["S_T", "ALL"]}        # Alldata 版：S_T・ALL（全条件で学習した最終モデル。検証データはない）
DATA_VERSIONS = ["5data", "Alldata"]       # データの版（手順3d）
MAIN_DATA = "なし"            # 主に使う外れ値の扱い（外れ値なし）
TEMP_HOLDOUT = 210            # S_T でテストにする設定温度 [℃]
E_PRIOR = {"Eh": (-40.0, 40.0), "Ev": (-20.0, 40.0)}   # E の事前分布（一様分布の範囲）[kJ/mol]
E_START = {"B": (7.0, 16.0), "A": (0.0, 7.0)}           # 手順5 の結果（best_E.csv）がないときの、E の出発点 [kJ/mol]
N_WARM, N_SAMP, CHAINS = 5000, 20000, 4    # 調整の反復数（捨てる）、本番の反復数、チェーン数（1 回の評価が約 1 ミリ秒なので、手順3c より多くできる）
N_SAMP_MAX = 100000           # R-hat が 1.01 を超えたときに増やす本番の回数の上限
N_JOBS = 8                    # 並列に回す数（チェーンと最尤推定。CPU の数に合わせる）
N_PRED_DRAWS = 200            # 予測に使う事後サンプル数（間引き後）
USE_SAVED_SAMPLES = True      # 保存済みの事後サンプルがあれば読み込む（サンプリングを省く）
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル1.5：ライブラリ、フォント、出力先
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
from joblib import Parallel, delayed
import arviz as az   # 収束診断（R-hat・有効サンプル数）に使う

OUT_DIR = os.path.join(REPO, "results", "step5b_mcmc")
STEP5_DIR = os.path.join(REPO, "results", "step5")      # 手順5 の結果（最良の E を読む）
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
print("Python:", sys.executable, "/", sys.version.split()[0])
print("ArviZ :", az.__version__)
print("出力先:", os.path.abspath(OUT_DIR))

### 共通の部品

データの読み込み、問題データ一覧の「除外」、評価、データの分け方、列を選んで当てるカーネル `OnDims` など。中身は `04_補助スクリプト/nb_common.py` と同じ。

In [ ]:
# セル：共通の部品（読み込み・モデル・評価）。04_補助スクリプト/nb_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Hyperparameter, Kernel, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                 # ノイズ分散の下限（CLAUDE.md §5。旧ノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 5)                # GPR の最適化のやり直し回数
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))   # リポジトリ直下（ノートは 01_機械学習コードフォルダ にあるので ".."。スクリプトは環境変数 NB_REPO で指定）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)     # 問題データ一覧の「除外」を使うか
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限への張り付きは kernels.csv に記録する

# ---------------------------------------------------------------- データの場所
DATA_DIR = os.path.join(REPO, "02_機械学習データ", "NR500")
TEST_DIR = os.path.join(REPO, "03_予測検証用テストデータ", "NR500")
PROBLEM_CSV = os.path.join(REPO, "02_機械学習データ", "問題データ一覧.csv")
MATS = {"PP": "PP", "MIX": "粉砕材混合PP"}
FILES = {   # (材料, 組): (上位フォルダ, 中間フォルダ, フォルダ, ファイル名)。どれも全条件のファイル（問題のある条件は、一覧で除く）
    ("PP", "5data"): (DATA_DIR, "02_バージンPP再々実験_5data", "03_pp_max_5data", "結合_03_pp_max_5data_外れ値{o}.csv"),
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "02_pp_max_3data", "結合_02_pp_max_3data_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "01_pp_max_Alldata", "結合_01_pp_max_Alldata_外れ値{o}.csv"),
    ("MIX", "5data"): (DATA_DIR, "04_粉砕材混合PP実験_5data", "03_mixpp_max_5data", "結合_03_mixpp_max_5data_外れ値{o}.csv"),
    ("MIX", "3data"): (TEST_DIR, "04_粉砕材混合PP実験_5data用検証データ(3data)", "02_mixpp_max_3data", "結合_02_mixpp_max_3data_外れ値{o}.csv"),
    ("MIX", "Alldata"): (DATA_DIR, "03_粉砕材混合PP実験_alldata", "01_mixpp_max_Alldata", "結合_01_mixpp_max_Alldata_外れ値{o}.csv"),
}
LEGACY_FILES = {   # 1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）。一覧の「除外」で同じデータになるかの確認用
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "05_pp_max_3data(no1.5-3.0)", "結合_05_pp_max_3data(no1.5-3.0)_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "04_pp_max_Alldata(no1.5-3.0)", "結合_04_pp_max_Alldata(no1.5-3.0)_外れ値{o}.csv"),
}
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "加熱シリンダ温度[℃]": "Tset", "流動中樹脂温度(℃)": "Tff"}
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm\.csv$"
D = {}                  # (材料, 組, 外れ値) → DataFrame。ノートで D = load_all() として入れる
REMOVED = {}            # (材料, 組, 外れ値) → 一覧の「除外」で除いた行数


def path_of(mat, grp, o, files=None):
    top, d1, d2, fn = (files or FILES)[(mat, grp)]
    return os.path.join(top, d1, d2, f"外れ値{o}", fn.format(o=o))


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MATS[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":                      # 空欄は条件全体。数字なら、そのショット（元ファイル名の先頭 3 桁）だけ
            m &= (df["shot"].to_numpy() == int(r["ショット"])) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


def first_shots(mat, grp):
    """条件ごとの、最初のショットの番号（外れ値ありのファイル。ショットが除かれる前の番号）。ショットの順番 pos の基準に使う"""
    ref = pd.read_csv(path_of(mat, grp, "あり"), encoding="utf-8-sig").rename(columns=COLS)
    ref["shot0"] = ref["file"].str.extract(RE_SRC)[0].astype(int)
    ref["Tset"] = ref["Tset"].astype(float)
    return ref.groupby(["Tset", "H", "v"], as_index=False)["shot0"].min()


def load(mat, grp, o, plist=None, exclude=None):
    """結合ファイルを読み、形式を確かめ、問題データ一覧の「除外」を除く。列 sens には、「感度確認」の ID が入る（なければ空文字）。
    列 shot は元ファイル名の先頭 3 桁（設定温度・厚みごとの通し番号で、同じ条件のショットは連続した番号）。列 pos は、その条件の中での順番（最初のショット = 1。外れ値で除かれたショットの番号は欠ける）"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    df = pd.read_csv(path_of(mat, grp, o), encoding="utf-8-sig")
    assert list(df.columns) == list(COLS), f"列名が想定と違う: {list(df.columns)}"
    df = df.rename(columns=COLS)
    assert df.isna().sum().sum() == 0, "欠損がある"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), "ファイル名の形式に合わない行がある"
    assert np.allclose(p[1].astype(float), df["Tset"]) and np.allclose(p[2].astype(float), df["v"]) \
        and np.allclose(p[3].astype(float), df["H"]), "ファイル名と列の値が一致しない"
    df["shot"] = p[0].astype(int)
    df["Tset"] = df["Tset"].astype(float)
    df = df.merge(first_shots(mat, grp), on=["Tset", "H", "v"], how="left")
    assert df["shot0"].notna().all(), "外れ値ありのファイルにない条件がある"
    df["pos"] = df["shot"] - df["shot0"].astype(int) + 1
    df = df.drop(columns="shot0")
    plist = read_problem_list() if plist is None else plist
    n0 = len(df)
    if exclude:
        drop, _ = problem_mask(df, mat, grp, plist, ("除外",))
        df = df[~drop].reset_index(drop=True)
    REMOVED[(mat, grp, o)] = n0 - len(df)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"])
    df["ln_P31"], df["ln_P32"] = np.log(df["P31"] * 1e6), np.log(df["P32"] * 1e6)   # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_v"] = np.log(df["v"])
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)          # 流動先端温度 [K]
    for c in ["gd", "Tff", "P32"]:   # 条件ごとの平均（成形条件から見たばらつきを見るモデル用。cm は condition mean）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    return df


def load_all(exclude=None):
    """全材料・全組・外れ値なし/あり を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp, o): load(mat, grp, o, plist, exclude) for mat in MATS for grp in ["5data", "3data", "Alldata"] for o in ["なし", "あり"]}
    for mat in MATS:
        for o in ["なし", "あり"]:
            assert not (set(out[(mat, "5data", o)]["cond"]) & set(out[(mat, "3data", o)]["cond"])), "学習と検証で条件が重なっている"
    return out


def legacy_check(D_):
    """全条件のファイルから一覧で除いたデータが、1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）と同じ行か（PP の 3data・Alldata）。
    no1.5-3.0 のファイルにも、1.5 mm・3 mm/s 以外の一覧の「除外」（P03 など）を同じように適用してから比べる"""
    rows = []
    plist = read_problem_list()
    plist = plist[~((plist["厚み"] == "1.5") & (plist["射出速度"] == "3") & (plist["設定温度"] == ""))]      # no1.5-3.0 のファイルで、すでに除かれている分
    for (mat, grp), _ in LEGACY_FILES.items():
        for o in ["なし", "あり"]:
            old = pd.read_csv(path_of(mat, grp, o, LEGACY_FILES), encoding="utf-8-sig")
            o2 = old.rename(columns=COLS)
            o2["shot"] = o2["file"].str.extract(RE_SRC)[0].astype(int)
            o2["Tset"] = o2["Tset"].astype(float)
            drop, _ = problem_mask(o2, mat, grp, plist, ("除外",)) if USE_EXCLUSIONS else (np.zeros(len(o2), bool), None)
            old = old[~drop]
            same = set(old["元ファイル"]) == set(D_[(mat, grp, o)]["file"])
            rows.append({"材料": MATS[mat], "組": grp, "外れ値": o, "一覧で除いた後の行数": len(D_[(mat, grp, o)]), "no1.5-3.0 の行数": len(old), "同じ行": same})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- モデル
def kernel_for(d, linear=False):
    """ARD Matern 5/2 ＋ ノイズ。linear=True なら、線形カーネル（DotProduct）も足す（低速側への外挿のため。CLAUDE.md §5）"""
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


class OnDims(Kernel):
    """入力の一部の列 dims にだけ当てるカーネル（足し算のカーネル m(…) + c(…) を作るための部品。手順5）。
    ハイパーパラメータは、中のカーネルのものをそのまま使う（scikit-learn の Exponentiation と同じ作り）。tag は、どの部品かの印（"m"・"c" など）"""

    def __init__(self, kernel, dims, tag=""):
        self.kernel = kernel
        self.dims = dims                                   # 列番号のリスト（scikit-learn の clone の決まりで、受け取ったものをそのまま持つ）
        self.tag = tag

    def get_params(self, deep=True):
        params = dict(kernel=self.kernel, dims=self.dims, tag=self.tag)
        if deep:
            params.update(("kernel__" + k, val) for k, val in self.kernel.get_params().items())
        return params

    @property
    def hyperparameters(self):
        return [Hyperparameter("kernel__" + h.name, h.value_type, h.bounds, h.n_elements) for h in self.kernel.hyperparameters]

    @property
    def theta(self):
        return self.kernel.theta

    @theta.setter
    def theta(self, theta):
        self.kernel.theta = theta

    @property
    def bounds(self):
        return self.kernel.bounds

    def __eq__(self, b):
        return type(self) == type(b) and self.dims == b.dims and self.kernel == b.kernel

    def __hash__(self):
        return hash((tuple(self.dims), self.tag))

    def __call__(self, X, Y=None, eval_gradient=False):
        Xd = np.asarray(X)[:, self.dims]
        Yd = None if Y is None else np.asarray(Y)[:, self.dims]
        return self.kernel(Xd, Yd, eval_gradient=eval_gradient)

    def diag(self, X):
        return self.kernel.diag(np.asarray(X)[:, self.dims])

    def is_stationary(self):
        return self.kernel.is_stationary()

    def __repr__(self):
        return f"OnDims({self.tag}, dims={self.dims}, {self.kernel!r})"


def make_gpr(d, linear=False, restarts=None):
    return GaussianProcessRegressor(kernel=kernel_for(d, linear), n_restarts_optimizer=N_RESTARTS if restarts is None else restarts, random_state=SEED)


def fit_ols(X_tr, y_tr):
    """切片ありの最小二乗回帰。係数、(XᵀX)⁻¹、残差分散を返す"""
    A_tr = np.column_stack([np.ones(len(X_tr)), X_tr])
    beta, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    resid = y_tr - A_tr @ beta
    return beta, np.linalg.inv(A_tr.T @ A_tr), resid @ resid / (len(y_tr) - A_tr.shape[1])


def predict_ols(model, X):
    """予測平均と、新しい1ショットに対する予測標準偏差"""
    beta, XtX_inv, s2 = model
    A_new = np.column_stack([np.ones(len(X)), X])
    lev = np.einsum("ij,jk,ik->i", A_new, XtX_inv, A_new)
    return A_new @ beta, np.sqrt(s2 * (1.0 + lev))


MODELS = {}             # 記号 → {"kind": "ols" か "gpr", "cols": [入力の列], "linear": 線形カーネルを足すか}。ノートで決める


def fit_model(key, tr, restarts=None):
    spec = MODELS[key]
    X, y = tr[spec["cols"]].to_numpy(), tr["ln_eta"].to_numpy()
    if spec["kind"] == "ols":
        return {"kind": "ols", "model": fit_ols(X, y)}
    sc = StandardScaler().fit(X)              # 標準化と中心化は学習データだけで計算する
    y0 = y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False), restarts).fit(sc.transform(X), y - y0)
    return {"kind": "gpr", "gp": gp, "sc": sc, "y0": y0}


def predict_model(key, fit, te):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X = te[MODELS[key]["cols"]].to_numpy()
    if fit["kind"] == "ols":
        mu, sd = predict_ols(fit["model"], X)
        return mu, sd, np.full(len(X), np.nan), float(np.sqrt(fit["model"][2]))
    mu, sd = fit["gp"].predict(fit["sc"].transform(X), return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"], sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD（各ショットの予測分布の負の対数密度の平均）、予測標準偏差の平均"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)),
            "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)), "mean_sd": float(np.mean(sd))}


def kernel_row(key, fit):
    """学習した GPR のハイパーパラメータ（振幅、長さスケール、ノイズ、線形項、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    k = gp.kernel_
    lin = MODELS[key].get("linear", False)
    prod, lin_prod = (k.k1.k1, k.k1.k2) if lin else (k.k1, None)
    ls = np.atleast_1d(prod.k2.length_scale)
    row = {"amplitude": float(np.sqrt(prod.k1.constant_value)), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "LML": float(gp.log_marginal_likelihood_value_),
           "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05),
           "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin_prod is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin_prod.k1.constant_value)), "lin_sigma0": float(lin_prod.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(MODELS[key]["cols"], ls)})
    return row


def within_stats(df):
    """同じ条件内のばらつき（プール）。dof：自由度、var_after_gd：ln γ̇ で説明した残りの分散"""
    g = df.groupby("cond")
    dof = int((g.size() - 1).sum())
    de = df["ln_eta"] - g["ln_eta"].transform("mean")
    dg = df["ln_gd"] - g["ln_gd"].transform("mean")
    var_e = float((de ** 2).sum() / dof)
    b = float((de * dg).sum() / (dg ** 2).sum())
    return {"dof": dof, "var_ln_eta": var_e, "sd_ln_eta": var_e ** 0.5, "corr_with_ln_gd": float((de * dg).sum() / np.sqrt((de ** 2).sum() * (dg ** 2).sum())),
            "slope_vs_ln_gd": b, "var_after_gd": float(((de - b * dg) ** 2).sum() / (dof - 1))}


# ---------------------------------------------------------------- データの分け方
def split_V(mat, o):
    """5data で学習し、3data で検証する（射出速度の両端の外挿）。te の group 列は、低速側（3・5 mm/s）か高速側（280・320 mm/s）"""
    tr, te = D[(mat, "5data", o)].copy(), D[(mat, "3data", o)].copy()
    te["group"] = np.where(te["v"] <= 5, "低速側", np.where(te["v"] >= 280, "高速側", "その他"))
    assert (te["group"] != "その他").all(), "検証データに、低速側・高速側のどちらでもない条件がある"
    return tr, te


def split_random(df):
    """行単位のランダム 20%（従来の分け方。random_state = SEED）"""
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    te = df.iloc[idx_te].copy()
    te["group"] = "全体"
    return df.iloc[idx_tr].copy(), te


def split_T(mat, o, t_test=210):
    """設定温度 t_test の全条件をテストにし、5data の残りの温度で学習する（温度の内挿の検証）。3data の同じ温度は te3 として別に返す"""
    d5, d3 = D[(mat, "5data", o)], D[(mat, "3data", o)]
    tr, te = d5[d5["Tset"] != t_test].copy(), d5[d5["Tset"] == t_test].copy()
    te["group"] = "全体"
    te3 = d3[d3["Tset"] == t_test].copy()
    te3["group"] = np.where(te3["v"] <= 5, "低速側", "高速側")
    return tr, te, te3


def pool_of(mat, o):
    """5data と 3data をまとめたデータ"""
    return pd.concat([D[(mat, "5data", o)], D[(mat, "3data", o)]], ignore_index=True)


# ---------------------------------------------------------------- 学習と評価の記録
METRICS, KERNELS, OLSROWS, PRED, FITS = [], [], [], {}, {}      # FITS：(材料, split, データ, モデル) → 学習した予測器
SENS_SUFFIX = "（感度確認の条件を除く）"


def _record(mat, name, data, key, fit, tr, te, pure):
    """学習した予測器 fit で te を予測し、指標（subset ごと）と予測を記録する"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in te["group"].unique():
        if g != "全体":
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    y_te = te["ln_eta"].to_numpy()
    mu, sd, sd_f, noise_sd = predict_model(key, fit, te)
    PRED[(mat, name, data, key)] = pd.DataFrame({"y": y_te, "mu": mu, "sd": sd, "sd_f": sd_f, "v": te["v"].to_numpy(), "H": te["H"].to_numpy(),
                                                 "Tset": te["Tset"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(),
                                                 "is_sens": te["is_sens"].to_numpy()})
    for sub, mask in subsets.items():
        if mask.sum() == 0:
            continue
        ev = evaluate(y_te[mask], mu[mask], sd[mask])
        METRICS.append({"material": MATS[mat], "split": name, "data": data, "model": key, "subset": sub, "n_train": len(tr),
                        "n_test": int(mask.sum()), "n_cond_test": int(te.loc[mask, "cond"].nunique()), **ev,
                        "mean_sd_latent": float(np.nanmean(sd_f[mask])) if fit["kind"] == "gpr" else np.nan, "noise_sd": noise_sd})


def run_split(mat, split, data, tr, te, keys=None, tag="", extra_te=None):
    """全モデルを学習し、指標（subset ごと）・カーネル・予測を記録する。te には group 列がある。
    te に「感度確認」の条件があれば、各 subset から除いた指標（subset 名の後ろに SENS_SUFFIX）も記録する。
    tag は split 名の後ろに付ける印。extra_te（{split 名: テスト}）があれば、同じ学習で、それらも評価する"""
    keys = keys or ALL_KEYS
    name = split + tag
    tests = {name: te, **{s + tag: t for s, t in (extra_te or {}).items()}}
    pure = within_stats(tr)
    for key in keys:
        t0 = time.time()
        fit = fit_model(key, tr)
        FITS[(mat, name, data, key)] = fit
        for nm, te_ in tests.items():
            _record(mat, nm, data, key, fit, tr, te_, pure)
        base = {"material": MATS[mat], "split": name, "data": data, "model": key, "n_train": len(tr)}
        if fit["kind"] == "gpr":
            KERNELS.append({**base, **kernel_row(key, fit), "pure_sd_ln_eta": pure["sd_ln_eta"], "pure_sd_after_gd": pure["var_after_gd"] ** 0.5})
        else:
            names = ["intercept"] + MODELS[key]["cols"]
            OLSROWS.append({**base, "resid_sd": float(np.sqrt(fit["model"][2])), **dict(zip(names, map(float, fit["model"][0])))})
        print(f"{MATS[mat]} {name} {data} {key}: 学習 {len(tr)} 行、{time.time() - t0:.1f} 秒")


# ---------------------------------------------------------------- 条件を1つずつ抜く交差検証（LOCO）
def loco_gpr(key, pool):
    """GPR：ハイパーパラメータを全データで決めて固定し、条件を抜いた予測を閉じた式で計算する（平均 = y_g − [K⁻¹_gg]⁻¹(K⁻¹y)_g、共分散 = [K⁻¹_gg]⁻¹）"""
    spec = MODELS[key]
    X, y = pool[spec["cols"]].to_numpy(), pool["ln_eta"].to_numpy()
    sc = StandardScaler().fit(X)
    Xs, yc = sc.transform(X), y - y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))              # ノイズ込みの共分散行列の逆行列
    alpha = Kinv @ yc
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        Minv = np.linalg.inv(Kinv[np.ix_(idx, idx)])
        mu[idx] = y[idx] - Minv @ alpha[idx]
        sd[idx] = np.sqrt(np.diag(Minv))
    return mu, sd, gp


def loco_ols(key, pool):
    """線形回帰：条件ごとに、その条件を除いて当てはめ直す"""
    cols = MODELS[key]["cols"]
    X, y = pool[cols].to_numpy(), pool["ln_eta"].to_numpy()
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        keep = np.ones(len(y), bool)
        keep[idx] = False
        mu[idx], sd[idx] = predict_ols(fit_ols(X[keep], y[keep]), X[idx])
    return mu, sd


# ---------------------------------------------------------------- 外れ値の検知（規則 f）と、データの版（5data 版・Alldata 版）
OUTLIER_Z = globals().get("OUTLIER_Z", 3.5)             # 規則 f：|z| がこれを超え、かつ |残差|（ln η）が次の値を超える条件を候補にする
OUTLIER_RESID = globals().get("OUTLIER_RESID", 0.1)     # 0.1 は粘度で約 10%


def cond_means(df):
    """条件ごとの平均（ln η と ln γ̇ をショットで平均した点）"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"), n=("ln_eta", "size")).reset_index()
    t["ln_H"] = np.log(t["H"])
    return t


def outlier_scan(df, z_thr=None, resid_thr=None):
    """規則 f：条件平均が、ほかの条件から予測した値と大きく違う条件を拾う。
    条件平均の GPR（入力 ln γ̇（条件平均）・ln H・Tset。R3 と同じ形）の、条件を 1 つずつ抜いた予測（閉じた式）との残差を、予測標準偏差で割って z とする。
    返す表の列：cond, Tset, H, v, n（ショット数）, resid（ln η の実測 − 予測）, sd, z, flag（|z| と |resid| が両方しきい値を超えたか）"""
    z_thr = OUTLIER_Z if z_thr is None else z_thr
    resid_thr = OUTLIER_RESID if resid_thr is None else resid_thr
    t = cond_means(df)
    X, y = t[["ln_gd", "ln_H", "Tset"]].to_numpy(), t["ln_eta"].to_numpy()
    Xs, yc = StandardScaler().fit_transform(X), y - y.mean()
    gp = make_gpr(3, True).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))
    d = np.diag(Kinv)
    t["resid"], t["sd"] = (Kinv @ yc) / d, np.sqrt(1.0 / d)           # 条件 i を抜いた予測：平均 = y_i − (K⁻¹y)_i / K⁻¹_ii、分散 = 1 / K⁻¹_ii
    t["z"] = t["resid"] / t["sd"]
    t["flag"] = (t["z"].abs() > z_thr) & (t["resid"].abs() > resid_thr)
    return t[["cond", "Tset", "H", "v", "n", "resid", "sd", "z", "flag"]]


DATA_VERSIONS = globals().get("DATA_VERSIONS", ["5data", "Alldata"])   # 検討するデータの版。"5data"：5data で学習し、3data で外挿を検証する版／"Alldata"：全条件で学習する版
CV_K = globals().get("CV_K", 10)                                       # Alldata 版の交差検証の分割数（条件単位）


def split_cv(pool, k=None):
    """条件単位の k 分割交差検証。同じ条件のショットは同じ側に入れる（SEED で条件を並べ替える）。(学習, テスト) の組の一覧を返す"""
    k = CV_K if k is None else k
    conds = np.array(sorted(pool["cond"].unique()))
    np.random.RandomState(SEED).shuffle(conds)
    out = []
    for i in range(k):
        te_c = set(conds[i::k])
        te = pool[pool["cond"].isin(te_c)].copy()
        te["group"] = "全体"
        out.append((pool[~pool["cond"].isin(te_c)].copy(), te))
    return out


def version_sets(mat, version, data="なし", t_test=210):
    """データの版ごとの分割を返す。{"pool": 最終モデル（ALL）の学習データ, "splits": {分割の名前: (学習, {集計の名前: テスト})}}
    5data 版：S_V（5data → 3data。低速側・高速側）、S_T（190・230 ℃ → 210 ℃。5data の 210 ℃ と 3data の 210 ℃ を別に集計）、CV（5data の中の条件単位の CV_K 分割）。最終モデルは 5data（3data は検証専用）。
    Alldata 版：S_T（Alldata の 190・230 ℃ で学習。テストは 5data 版と同じ行：5data ファイルの 210 ℃ と 3data ファイルの 210 ℃。そのまま比べられる）、
    S_Vall（Alldata のうち 5data の条件だけで学習 → 3data の条件を検証。PP のみ意味がある〔粉砕材混合 PP は 5data と同じ〕）、CV（Alldata の条件単位の CV_K 分割）。最終モデルは Alldata（全条件）"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if version == "5data":
        tr_v, te_v = split_V(mat, data)
        tr_t, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_V": (tr_v, {"S_V": te_v}), "S_T": (tr_t, {"S_T": te_t, "S_T3": te_t3})}
        pool = d5
    else:
        assert version == "Alldata", version
        pool = D[(mat, "Alldata", data)]
        _, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_T": (pool[pool["Tset"] != t_test].copy(), {"S_T": te_t, "S_T3": te_t3})}
        te_v = d3.copy()
        te_v["group"] = np.where(te_v["v"] <= 5, "低速側", np.where(te_v["v"] >= 280, "高速側", "その他"))
        sp["S_Vall"] = (pool[pool["cond"].isin(set(d5["cond"]))].copy(), {"S_Vall": te_v})
    for i, (tr_i, te_i) in enumerate(split_cv(pool), 1):
        sp[f"CV{i}"] = (tr_i, {f"CV{i}": te_i})
    return {"pool": pool, "splits": sp}


# ---------------------------------------------------------------- 判定の表示（GPR6月ノートの「判定」にならう）
def judge(ok, text):
    """結果の判定を1行で表示する。ok が True なら「OK」、False なら「要確認」"""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{text}")


# ---------------------------------------------------------------- 高解像度のグラフ（GPR6月ノートのセル9-2 にならう。報告書・発表資料用）
HR_COLORS = {"H": {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"},       # 厚みごとの色（6月のノートと同じ並び。黄は、線で見やすい濃さにした）
             "Tset": {190: "#1f77b4", 210: "#2ca02c", 230: "#d62728"}}                       # 設定温度ごとの色
HR_MARKS = {"H": {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}, "Tset": {190: "o", 210: "s", 230: "^"}}
HR_DEFAULT = dict(layouts=["thickness", "temperature"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                  xlim={"thickness": None, "temperature": (30, 50000)}, ylim={"thickness": None, "temperature": (0.5, 5000)}, figsize=(15, 10), dpi=600, font={"axis": 24, "legend": 24, "title": 24, "tick": 28},
                  marker=200, marker_shot=70, line=3.0, show_title=True, show_xlabel=False, show_ylabel=False, formats=["png"], show=False, n_grid=120)


def hr_split_conds(mat, data, split, t_test=210):
    """学習に使った条件の集合。S_V：5data／S_T：5data の t_test 以外の温度／ALL：5data と 3data"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if split == "S_V":
        return set(d5["cond"])
    if split == "S_T":
        return set(d5.loc[d5["Tset"] != t_test, "cond"])
    return set(d5["cond"]) | set(d3["cond"])


def hr_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。Tff・P3_2・射出速度は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（どの入力のモデルでも描けるように）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), tff=("ln_Tff", "mean"), p32=("ln_P32", "mean"), lv=("ln_v", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    return pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": ip("tff"), "ln_Tff_cm": ip("tff"),
                         "ln_P32": ip("p32"), "ln_P32_cm": ip("p32"), "ln_v": ip("lv")})


HR_ALL = globals().get("HR_ALL", True)                 # すべての図を、高解像度でも保存するか（True：out_dir/highres/ に HR_ALL_DPI で保存。Git の管理外）
HR_ALL_DPI = globals().get("HR_ALL_DPI", 600)


def save_fig(fig, out_dir, stem):
    """図を out_dir に PNG で保存する（解像度は FIG_DPI）。HR_ALL が True なら、同じ図を HR_ALL_DPI（既定 600 dpi）で out_dir/highres/ にも保存する。保存したファイルの一覧を返す"""
    os.makedirs(out_dir, exist_ok=True)
    paths = [os.path.join(out_dir, f"{stem}.png")]
    fig.savefig(paths[0], dpi=globals().get("FIG_DPI", 200), bbox_inches="tight")
    if HR_ALL:
        os.makedirs(os.path.join(out_dir, "highres"), exist_ok=True)
        paths.append(os.path.join(out_dir, "highres", f"{stem}.png"))
        fig.savefig(paths[1], dpi=HR_ALL_DPI, bbox_inches="tight")
    return paths


def hr_flow(predict, mat, data, train_conds, title_model, stem, out_dir, pool=None, **kw):
    """せん断速度 ― 粘度の学習結果グラフを、高解像度で 1 枚ずつ保存する（6月のノートのセル9-2 と同じ見せ方）。
    predict(Xg) → (予測平均 ln η, 予測標準偏差（ノイズ込み）, 潜在関数の標準偏差)。Xg は hr_inputs の DataFrame。
    train_conds：学習に使った条件の集合（それ以外は、検証の点として白抜きで重ねる）。pool：点と曲線の範囲に使うデータ（None なら 5data＋3data）。
    設定は HR_DEFAULT を参照（kw で上書き）。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**HR_DEFAULT, **kw}
    os.makedirs(out_dir, exist_ok=True)
    pool = pool_of(mat, data) if pool is None else pool
    curves = {}                                                         # (厚み, 設定温度) → (x, 予測平均, 帯の幅の標準偏差)。レイアウトが違っても同じ線なので、1 回だけ計算する
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf = predict(hr_inputs(sub, lg, h, t))
        curves[(float(h), int(t))] = (np.exp(lg), np.asarray(mu), np.asarray(sd if cfg["band"] == "predict" else sdf))
    saved = []
    ft = cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout in cfg["layouts"]:
        by, sk = ("H", "Tset") if layout == "thickness" else ("Tset", "H")        # by：1 枚ごとに固定する量、sk：1 枚の中で線を分ける量
        for gv in sorted({k[0] if by == "H" else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"], dpi=cfg["dpi"])
            series, any_test, any_sens, xs, ys = [], False, False, [], []
            for sv in sorted({(k[1] if by == "H" else k[0]) for k in curves if (k[0] if by == "H" else k[1]) == gv}):
                h, t = (gv, sv) if by == "H" else (sv, gv)
                col, mk = HR_COLORS[sk][sv], HR_MARKS[sk][sv]
                x, mu, s = curves[(float(h), int(t))]
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.22, zorder=15)
                    ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                    ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                ax.plot(x, np.exp(mu), color=col, lw=cfg["line"], zorder=20)
                sub = pool[(np.isclose(pool["H"], h)) & (pool["Tset"] == t)]
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, tr_m, se_m, msize = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["cond"].isin(train_conds).to_numpy(), a["sens"].to_numpy().astype(bool), cfg["marker"]
                else:
                    px, py, tr_m, se_m, msize = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["cond"].isin(train_conds).to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[tr_m], py[tr_m], s=msize, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14 - list(HR_COLORS[sk]).index(sv))
                if cfg["show_test"] and (~tr_m).any():
                    ax.scatter(px[~tr_m], py[~tr_m], s=msize, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14 - list(HR_COLORS[sk]).index(sv))
                    any_test = True
                if cfg["mark_sens"] and se_m.any():
                    ax.scatter(px[se_m], py[se_m], s=msize * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if sk == "H" else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=col, lw=cfg["line"], marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.6,
                               label=(f"{sv} mm" if sk == "H" else f"{int(sv)} ℃")) for sv, col, mk in series]
            if any_test:
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]), markerfacecolor="white", markeredgewidth=2.5, label="検証の条件（白抜き）"))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]), markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout] if isinstance(cfg["xlim"], dict) else cfg["xlim"]       # 範囲：(下限, 上限)＝固定／None＝その図のデータに合わせる
            yl = cfg["ylim"][layout] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            auto_x, auto_y = xl is None, yl is None
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if auto_x else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if auto_y else yl))
            for axis, auto in ((ax.xaxis, auto_x), (ax.yaxis, auto_y)):
                if auto:                               # 範囲が狭いので、1・2・5 × 10^n に目盛りを付けて、普通の数で表す
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                else:                                  # 6月のノートと同じ（10 の累乗の目盛り）
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv} mm" if by == "H" else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MATS[mat]}　せん断速度と粘度の関係（{gtxt}）\n{title_model}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'by-thickness' if by == 'H' else 'by-temperature'}_{(str(gv) + 'mm') if by == 'H' else (str(int(gv)) + 'C')}"
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=cfg["dpi"])
                saved.append(path)
            if cfg["show"]:
                plt.show()
            plt.close(fig)
    return saved

### 手順5 の部品

構造と H の扱い（`layout5`・`design5`・`kernel5`）、学習と予測（`fit5`・`predict5`）、条件平均の表（`cond_table5`）など。中身は `04_補助スクリプト/step5_core.py` と同じ。

In [ ]:
# セル：手順5の部品（構造・H の扱い・プロファイル尤度・P3_2 の予測・学習の仕事）。04_補助スクリプト/step5_core.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import time as _time

from scipy.linalg import cho_solve

R_GAS, TREF_K = 8.314, 483.15                     # 気体定数 [J/mol/K]、基準温度 210 ℃ [K]
HV_JA = {"1": "(1) Hを残す", "2": "(2) m＋c(Fo)に分ける", "4": "(4) P3_2（Hなし）", "14": "(1+4)", "24": "(2+4)"}
STRUCT_JA5 = {"B": "B（横・縦を別々）", "A": "A（縦だけ）", "C": "C（構造なし）"}
COL_JA = {"u": "ln γ̇", "H": "ln H", "T": "Tset", "w": "ln Fo", "p": "ln P3_2"}


def shift_ln(Tk, E):
    """ln a(T; E) = (E/R)(1/T − 1/Tref)。E は kJ/mol、Tk は K"""
    return (E * 1000.0 / R_GAS) * (1.0 / np.asarray(Tk) - 1.0 / TREF_K)


def has_p(hv):
    return "4" in hv


def layout5(struct, hv):
    """入力の列の並び cols と、(2) の足し算の分け方 (m の列番号, c の列番号)（足し算でなければ None, None）。
    列の記号：u＝ln γ̇（B・A はシフト後）、H＝ln H、T＝Tset、w＝ln Fo、p＝ln P3_2（条件平均）"""
    if struct == "C":
        base = {"1": ["u", "H", "T"], "2": ["u", "T", "w"], "4": ["u", "T", "p"], "14": ["u", "H", "T", "p"], "24": ["u", "T", "p", "w"]}
    else:
        base = {"1": ["u", "w"], "2": ["u", "w"], "4": ["u", "p"], "14": ["u", "w", "p"], "24": ["u", "p", "w"]}
    cols = base[hv]
    if hv in ("2", "24"):
        return cols, [i for i, c in enumerate(cols) if c != "w"], [cols.index("w")]
    return cols, None, None


def kernel5(struct, hv):
    """(1)(4)(1+4)：ARD Matern 5/2 ＋ 線形 ＋ ノイズ（kernel_for と同じ）。(2)(2+4)：m の部分と c の部分を足し算（それぞれ Matern ＋ 線形）＋ ノイズ"""
    cols, m, c = layout5(struct, hv)
    if m is None:
        return kernel_for(len(cols), True)

    def part(d):
        return C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5) + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))

    return OnDims(part(len(m)), m, "m") + OnDims(part(len(c)), c, "c") + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def design5(df, v):
    """データ → (標準化の定数を決めるための入力 X0、実際の入力 Xs、出力のシフト量 ysh)。v：{"struct", "hv", "Eh", "Ev", "pcol"}。
    入力の γ̇ は条件平均（ln_gd_cm）。B・A は u = ln γ̇ + ln a(Tset; E_h)、ysh = ln a(Tset; E_v)。P3_2 は pcol の列（既定 ln_P32_cm）"""
    cols = layout5(v["struct"], v["hv"])[0]
    lng, lnH, T = df["ln_gd_cm"].to_numpy(), df["ln_H"].to_numpy(), df["Tset"].to_numpy()
    raw = {"u": lng, "H": lnH, "T": T, "w": -(lng + 3.0 * lnH)}            # ln Fo = −(ln γ̇ + 3 ln H) + 定数（シフト前の γ̇ で計算）
    if has_p(v["hv"]):
        raw["p"] = df[v.get("pcol", "ln_P32_cm")].to_numpy()
    shifted = dict(raw)
    ysh = np.zeros(len(df))
    if v["struct"] != "C":
        Tk = T + 273.15
        shifted["u"] = lng + shift_ln(Tk, v["Eh"])
        ysh = shift_ln(Tk, v["Ev"])
    return np.column_stack([raw[c] for c in cols]), np.column_stack([shifted[c] for c in cols]), ysh


def fit5(v, tr, restarts, theta0=None):
    """学習。標準化・中心化は E に依存しない定数（シフト前の入力の平均・標準偏差、ln η の平均）。theta0：カーネルの初期値（前の格子点の解）"""
    X0, Xs, ysh = design5(tr, v)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    y = tr["ln_eta"].to_numpy()
    y0 = float(y.mean())
    k = kernel5(v["struct"], v["hv"])
    if theta0 is not None:
        k = k.clone_with_theta(theta0)
    gp = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=restarts, random_state=SEED).fit((Xs - mu_x) / sd_x, y - ysh - y0)
    return {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "v": v}


def rebuild5(v, tr, theta):
    """学習済みのカーネルの値 theta から、学習をやり直さずに予測器を作り直す（最適化なし。並列の仕事から theta だけを受け取る）"""
    X0, Xs, ysh = design5(tr, v)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    y = tr["ln_eta"].to_numpy()
    y0 = float(y.mean())
    gp = GaussianProcessRegressor(kernel=kernel5(v["struct"], v["hv"]).clone_with_theta(theta), optimizer=None).fit((Xs - mu_x) / sd_x, y - ysh - y0)
    return {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "v": v}


def predict5(fit, df):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X0, Xs, ysh = design5(df, fit["v"])
    mu, sd = fit["gp"].predict((Xs - fit["mu_x"]) / fit["sd_x"], return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"] + ysh, sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def component_predict(fit, df, part):
    """足し算のモデル (2)(2+4) の成分 part（"m" か "c"）の事後平均と事後標準偏差（ln η の単位。定数は m と c の間で入れ替えられる）"""
    gp = fit["gp"]
    k = gp.kernel_
    kc = k.k1.k1 if part == "m" else k.k1.k2
    assert kc.tag == part
    X0, Xs, ysh = design5(df, fit["v"])
    Xt = (Xs - fit["mu_x"]) / fit["sd_x"]
    Ks = kc(Xt, gp.X_train_)
    mean = Ks @ gp.alpha_
    var = kc.diag(Xt) - np.einsum("ij,ji->i", Ks, cho_solve((gp.L_, True), Ks.T))
    return mean, np.sqrt(np.maximum(var, 0.0))


def kernel_row5(fit, tr=None):
    """ハイパーパラメータの一覧（部品ごとの振幅・長さスケール・線形項、ノイズ、対数周辺尤度、下限・上限への張り付き）"""
    v, gp = fit["v"], fit["gp"]
    k = gp.kernel_
    cols = layout5(v["struct"], v["hv"])[0]
    if hasattr(k.k1.k1, "tag"):
        parts = [("m", k.k1.k1.kernel, k.k1.k1.dims), ("c", k.k1.k2.kernel, k.k1.k2.dims)]
    else:
        parts = [("f", k.k1, list(range(len(cols))))]
    row = {"Eh": v.get("Eh", np.nan), "Ev": v.get("Ev", np.nan), "LML": float(gp.log_marginal_likelihood_value_), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05)}
    at_bound = False
    for name, kk, dims in parts:
        prod, lin = kk.k1, kk.k2
        ls = np.atleast_1d(prod.k2.length_scale)
        at_bound = at_bound or bool(np.any(ls > 99) or np.any(ls < 0.0105))
        row[f"amp_{name}"] = float(np.sqrt(prod.k1.constant_value))
        row[f"lin_amp_{name}"] = float(np.sqrt(lin.k1.constant_value))
        row[f"lin_sigma0_{name}"] = float(lin.k2.sigma_0)
        for d, l in zip(dims, ls):
            row[f"ls_{name}_{cols[d]}"] = float(l)
    row["ls_at_bound"] = at_bound
    return row


def cond_table5(df):
    """条件ごとの平均（ln η、ln γ̇、ln P3_2）。尤度の比較に使う"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"), ln_P32=("ln_P32", "mean"),
              n=("ln_eta", "size"), is_sens=("is_sens", "max")).reset_index()
    t["ln_H"], t["ln_gd_cm"], t["ln_P32_cm"] = np.log(t["H"]), t["ln_gd"], t["ln_P32"]
    return t


# ---------------------------------------------------------------- プロファイル尤度
def prof_chain(cdf, base_v, points, restarts):
    """格子の点 points = [(E_h, E_v), …] を順に学習して LML を返す。前の点の解を初期値にする（局所解でがたつくのを減らす）"""
    warnings.simplefilter("ignore")
    out, theta = [], None
    for eh, ev in points:
        f = fit5({**base_v, "Eh": float(eh), "Ev": float(ev)}, cdf, restarts, theta)
        theta = f["gp"].kernel_.theta
        out.append((float(f["gp"].log_marginal_likelihood_value_), float(np.sqrt(f["gp"].kernel_.k2.noise_level))))
    return out


def chains_for(struct, grid, eh_fix=None):
    """構造ごとの格子を、順に計算する点の列（chain）に分ける。A：E_v の 1 列／B：E_h ごとに E_v の列（eh_fix が数なら、E_h をその値に固定した 1 列）"""
    if struct == "A":
        return [[(0.0, float(e)) for e in grid["A"]]]
    if eh_fix is not None:
        return [[(float(eh_fix), float(e)) for e in grid["B_ev"]]]
    return [[(float(eh), float(ev)) for ev in grid["B_ev"]] for eh in grid["B_eh"]]


def profile_best(pdf, key_cols):
    """格子の点の表 → 条件（key_cols）ごとの最良の E と、LML が最大から 1.92 以内の範囲"""
    rows = []
    for key, g in pdf.groupby(key_cols):
        r = g.loc[g["LML"].idxmax()]
        inside = g[g["LML"] >= r["LML"] - 1.92]
        z = g[(g["Eh"] == 0.0) & (g["Ev"] == 0.0)]
        row = dict(zip(key_cols, key))
        diff = inside["Ev"] - inside["Eh"]
        row.update({"Eh_best": float(r["Eh"]), "Ev_best": float(r["Ev"]), "diff_best": float(r["Ev"] - r["Eh"]), "LML_best": float(r["LML"]),
                    "LML_at_E0": float(z["LML"].iloc[0]) if len(z) else np.nan, "n_in_95": int(len(inside)),
                    "Eh_95_lo": float(inside["Eh"].min()), "Eh_95_hi": float(inside["Eh"].max()), "Ev_95_lo": float(inside["Ev"].min()), "Ev_95_hi": float(inside["Ev"].max()),
                    "diff_95_lo": float(diff.min()), "diff_95_hi": float(diff.max()), "n_points": int(len(g)),
                    "at_edge": bool(r["Ev"] in (g["Ev"].min(), g["Ev"].max()) or (g["Eh"].nunique() > 1 and r["Eh"] in (g["Eh"].min(), g["Eh"].max())))})
        row["dLML_vs_E0"] = row["LML_best"] - row["LML_at_E0"]
        rows.append(row)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- P3_2 の予測（(4) の「予測した P3_2」）
def p32_model(tr):
    """条件平均の ln P3_2 を (ln γ̇, ln H, Tset) から予測する GPR（学習の条件の点で学習）"""
    ct = cond_table5(tr)
    X = ct[["ln_gd_cm", "ln_H", "Tset"]].to_numpy()
    sc = StandardScaler().fit(X)
    y0 = float(ct["ln_P32_cm"].mean())
    gp = GaussianProcessRegressor(kernel=kernel_for(3, True), n_restarts_optimizer=N_RESTARTS, random_state=SEED).fit(sc.transform(X), ct["ln_P32_cm"].to_numpy() - y0)
    return {"gp": gp, "sc": sc, "y0": y0}


def p32_predict(pm, te):
    """テストの条件ごとに ln P3_2 を予測する。条件ごとの表（cond、実測の条件平均、予測平均、予測標準偏差〔ノイズ込み〕）と、行ごとの条件の番号を返す"""
    ct = cond_table5(te)
    mu, sd = pm["gp"].predict(pm["sc"].transform(ct[["ln_gd_cm", "ln_H", "Tset"]].to_numpy()), return_std=True)
    out = pd.DataFrame({"cond": ct["cond"], "H": ct["H"], "Tset": ct["Tset"], "v": ct["v"], "actual": ct["ln_P32_cm"], "pred": mu + pm["y0"], "pred_sd": sd})
    idx = pd.Series(np.arange(len(ct)), index=ct["cond"]).loc[te["cond"]].to_numpy()
    return out, idx


def predict5_p(fit, te, pm, n_draw, seed):
    """P3_2 を予測して使う場合：P3_2 の予測分布から n_draw 回引き、そのつど ln η を予測して、平均と分散にまとめる（全分散の法則）"""
    ptab, idx = p32_predict(pm, te)
    z = np.random.RandomState(seed).standard_normal((n_draw, len(ptab)))
    mus, vs, vfs = [], [], []
    for d in range(n_draw):
        pdraw = ptab["pred"].to_numpy() + ptab["pred_sd"].to_numpy() * z[d]
        mu, sd, sdf, _ = predict5(fit, te.assign(ln_P32_cm=pdraw[idx]))
        mus.append(mu)
        vs.append(sd ** 2)
        vfs.append(sdf ** 2)
    mus = np.array(mus)
    between = mus.var(axis=0)
    return mus.mean(axis=0), np.sqrt(np.mean(vs, axis=0) + between), np.sqrt(np.mean(vfs, axis=0) + between)


# ---------------------------------------------------------------- 学習と予測の仕事（並列）
def job5(mat, ver, sname, tr, tests, v, restarts, n_draw):
    """1 つのモデル（v）を学習し、tests の各テストを予測する。戻り値は、theta と予測（受け渡しやすいように、モデルそのものは返さない）。
    P3_2 を使うモデルは、実測の P3_2（条件平均）と、予測した P3_2 の両方で評価する"""
    warnings.simplefilter("ignore")
    t0 = _time.time()
    fit = fit5(v, tr, restarts)
    out = {"theta": fit["gp"].kernel_.theta, "kernel": kernel_row5(fit), "preds": {}}
    pm = p32_model(tr) if has_p(v["hv"]) else None
    for tn, te in tests.items():
        mu, sd, sdf, _ = predict5(fit, te)
        out["preds"][(tn, "実測" if pm is not None else "―")] = (mu, sd, sdf)
        if pm is not None:
            out["preds"][(tn, "予測")] = predict5_p(fit, te, pm, n_draw, SEED)
    out["sec"] = _time.time() - t0
    return out

### 手順5b の部品

条件ごとにまとめた尤度（`make_grouped`・`grouped_lml`）、予測（`grouped_predict`・`predict_mixture`）、対数事後密度と事前分布（`make_logpost`）、最尤推定（`fit_mle5`）、適応型メトロポリス法（`run_chain5`）、評価・収束診断（`evaluate_mixture`・`diag_summary`）。中身は `04_補助スクリプト/step5b_core.py` と同じ。

In [ ]:
# セル：手順5bの部品（条件ごとにまとめた尤度・MCMC・最尤推定・混合予測）。04_補助スクリプト/step5b_core.py の中身。直すときは、そのファイルを直して、ノートを作り直す
from scipy.linalg import LinAlgError, cholesky
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm

E_NAMES = {"B": ["Eh", "Ev"], "A": ["Ev"], "C": []}
TARGET_ACC = 0.234


def theta_names5(struct, hv):
    """MCMC で動かすパラメータの名前（カーネルの対数パラメータ → ずらし量 E）。順番は kernel5 の theta と同じ"""
    cols, m, c = layout5(struct, hv)
    names = []
    if m is None:
        names += ["log_c"] + [f"log_ls_{x}" for x in cols] + ["log_c_lin", "log_sigma0"]
    else:
        for part, idx in (("m", m), ("c", c)):
            names += [f"log_c_{part}"] + [f"log_ls_{part}_{cols[i]}" for i in idx] + [f"log_c_lin_{part}", f"log_sigma0_{part}"]
    names += ["log_noise"] + E_NAMES[struct]
    assert len(names) == len(kernel5(struct, hv).theta) + len(E_NAMES[struct]), "パラメータの名前の数が合わない"
    return names


def model_v(struct, hv, phi):
    """パラメータのベクトル phi（カーネルの対数パラメータ → E）から、構造の指定 v を作る"""
    ne = len(E_NAMES[struct])
    ev_ = list(phi[len(phi) - ne:]) if ne else []
    named = dict(zip(E_NAMES[struct], ev_))
    return {"struct": struct, "hv": hv, "pcol": "ln_P32_cm", "Eh": float(named.get("Eh", 0.0)), "Ev": float(named.get("Ev", 0.0))}


def make_grouped(tr, struct, hv):
    """学習データ（ショットの表）から、条件ごとにまとめた量を作る。標準化の定数と y0 は、手順5と同じくショットの行から計算する"""
    ct = cond_table5(tr)
    dev = tr["ln_eta"] - tr.groupby("cond")["ln_eta"].transform("mean")
    ss = (dev ** 2).groupby(tr["cond"], sort=False).sum().reindex(ct["cond"]).to_numpy()
    X0 = design5(tr, {"struct": struct, "hv": hv, "pcol": "ln_P32_cm", "Eh": 0.0, "Ev": 0.0})[0]
    return {"ct": ct, "n": ct["n"].to_numpy().astype(float), "ss": ss, "y0": float(tr["ln_eta"].mean()), "mu_x": X0.mean(axis=0), "sd_x": X0.std(axis=0),
            "N": int(len(tr)), "m": int(len(ct))}


def _train_parts(gd, v):
    X0, Xs, ysh = design5(gd["ct"], v)
    return (Xs - gd["mu_x"]) / gd["sd_x"], gd["ct"]["ln_eta"].to_numpy() - ysh - gd["y0"]


def grouped_lml(gd, v, kern, theta_k):
    """条件ごとにまとめた対数尤度（ショットごとの GPR の対数周辺尤度と一致する）。kern：kernel5 で作ったカーネル（theta を書き換える）"""
    kern.theta = theta_k
    Xt, yb = _train_parts(gd, v)
    noise = kern.k2.noise_level
    K = kern.k1(Xt)
    K[np.diag_indices_from(K)] += noise / gd["n"]
    try:
        L = cholesky(K, lower=True)
    except LinAlgError:
        return -np.inf
    alpha = cho_solve((L, True), yb)
    ll_cond = -0.5 * float(yb @ alpha) - float(np.log(np.diag(L)).sum()) - 0.5 * gd["m"] * np.log(2.0 * np.pi)
    ll_within = -0.5 * (gd["N"] - gd["m"]) * np.log(2.0 * np.pi * noise) - 0.5 * float(np.log(gd["n"]).sum()) - 0.5 * float(gd["ss"].sum()) / noise
    return ll_cond + ll_within


def grouped_predict(gd, struct, hv, phi, te, kern=None):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差。条件平均だけで厳密に計算できる（ショットごとの GPR の予測と一致する）"""
    kern = kern or kernel5(struct, hv)
    nk = len(kern.theta)
    kern.theta = phi[:nk]
    v = model_v(struct, hv, phi)
    Xt, yb = _train_parts(gd, v)
    noise = kern.k2.noise_level
    K = kern.k1(Xt)
    K[np.diag_indices_from(K)] += noise / gd["n"]
    L = cholesky(K, lower=True)
    alpha = cho_solve((L, True), yb)
    X0, Xs, yshT = design5(te, v)
    Xq = (Xs - gd["mu_x"]) / gd["sd_x"]
    Ks = kern.k1(Xq, Xt)
    var_f = np.maximum(kern.k1.diag(Xq) - np.einsum("ij,ji->i", Ks, cho_solve((L, True), Ks.T)), 0.0)
    return Ks @ alpha + gd["y0"] + yshT, np.sqrt(var_f + noise), np.sqrt(var_f)


def predict_mixture(gd, struct, hv, phis, te):
    """事後サンプル phis（行ごとに 1 サンプル）の予測分布を、等しい重みで混ぜる。戻り値：(成分ごとの平均, 成分ごとの標準偏差〔ノイズ込み〕, 混合の平均, 混合の標準偏差, 混合の潜在関数の標準偏差)"""
    kern = kernel5(struct, hv)
    out = [grouped_predict(gd, struct, hv, ph, te, kern) for ph in np.atleast_2d(phis)]
    mus, sds, sdf = (np.array([o[i] for o in out]) for i in range(3))
    mu = mus.mean(axis=0)
    return mus, sds, mu, np.sqrt((sds ** 2 + mus ** 2).mean(axis=0) - mu ** 2), np.sqrt((sdf ** 2 + mus ** 2).mean(axis=0) - mu ** 2)


def make_logpost(gd, struct, hv, e_prior):
    """対数事後密度の関数と、パラメータの範囲（事前分布）。範囲の中は一様（カーネルは対数スケール、E はそのまま）なので、範囲内では対数尤度そのもの"""
    kern = kernel5(struct, hv)
    nk = len(kern.theta)
    lo = np.r_[kern.bounds[:, 0], [e_prior[n][0] for n in E_NAMES[struct]]]
    hi = np.r_[kern.bounds[:, 1], [e_prior[n][1] for n in E_NAMES[struct]]]

    def logpost(phi):
        if np.any(phi <= lo) or np.any(phi >= hi):
            return -np.inf
        return grouped_lml(gd, model_v(struct, hv, phi), kern, phi[:nk])

    return logpost, lo, hi


def fit_mle5(gd, struct, hv, e_prior, e0, e_fixed, n_starts=8, seed=0):
    """最尤推定（同じ尤度を scipy の L-BFGS-B で最大化）。e_fixed が True なら E を e0 に固定してカーネルだけ、False なら E も動かす（e0 から出発）。"""
    logpost, lo, hi = make_logpost(gd, struct, hv, e_prior)
    ne = len(E_NAMES[struct])
    nk = len(lo) - ne
    rng = np.random.default_rng(seed)
    base = kernel5(struct, hv).theta
    best_x, best_f = None, np.inf
    free = np.arange(nk) if (e_fixed or ne == 0) else np.arange(nk + ne)

    def neg(x):
        phi = np.r_[x[:nk], np.asarray(e0, float)] if (e_fixed and ne) else np.array(x, float)
        v = logpost(phi)
        return -v if np.isfinite(v) else 1e10

    bnd = [(lo[i] + 1e-6, hi[i] - 1e-6) for i in free]
    for s in range(n_starts):
        th = np.clip(base if s == 0 else rng.uniform(lo[:nk] + 0.5, hi[:nk] - 0.5), lo[:nk] + 1e-3, hi[:nk] - 1e-3)
        e_start = np.asarray(e0, float) if (s == 0 or ne == 0) else np.asarray(e0, float) + rng.normal(0, 1.0, ne)
        x0 = th if (e_fixed or ne == 0) else np.r_[th, np.clip(e_start, lo[nk:] + 1e-3, hi[nk:] - 1e-3)]
        r = minimize(neg, x0, method="L-BFGS-B", bounds=bnd)
        if r.fun < best_f:
            best_x, best_f = r.x, r.fun
    phi = np.r_[best_x[:nk], np.asarray(e0, float)] if (e_fixed and ne) else np.array(best_x, float)
    return phi, -best_f


def run_chain5(gd, struct, hv, e_prior, phi0, init_sd, n_warm, n_samp, seed):
    """適応型ランダムウォーク・メトロポリス法の 1 チェーン（手順3c と同じ。調整中：提案 N(φ, λΣ) の Σ を足跡から学び、λ を採択率が 0.234 に近づくよう合わせる。本番：固定）"""
    rng = np.random.default_rng(seed)
    logpost, lo, hi = make_logpost(gd, struct, hv, e_prior)
    k = len(phi0)
    th = np.array(phi0, dtype=float)
    lp = logpost(th)
    cov, log_lam = np.diag(np.asarray(init_sd, float) ** 2), 0.0
    warm = np.empty((n_warm, k))
    draws, lps, acc = np.empty((n_samp, k)), np.empty(n_samp), np.zeros(n_warm + n_samp, dtype=bool)
    t0 = time.time()
    for it in range(n_warm + n_samp):
        L = np.linalg.cholesky(np.exp(log_lam) * cov)
        prop = th + L @ rng.standard_normal(k)
        lp_prop = logpost(prop)
        a = np.exp(min(0.0, lp_prop - lp)) if np.isfinite(lp_prop) else 0.0
        if rng.uniform() < a:
            th, lp, acc[it] = prop, lp_prop, True
        if it < n_warm:
            warm[it] = th
            log_lam += (it + 1) ** -0.6 * (a - TARGET_ACC)
            if it >= 199 and (it + 1) % 50 == 0 and it < 0.75 * n_warm:
                cov = np.cov(warm[(it + 1) // 2: it + 1].T) * (2.38 ** 2 / k) + np.eye(k) * 1e-10
        else:
            draws[it - n_warm], lps[it - n_warm] = th, lp
    return {"draws": draws, "lp": lps, "accepted": acc[n_warm:], "acc_warm": acc[:n_warm].mean(), "acc_samp": acc[n_warm:].mean(), "lam": np.exp(log_lam), "seconds": time.time() - t0}


def evaluate_mixture(y, mus, sds):
    """正規分布を等しい重みで混ぜた予測分布での指標。mus, sds は (成分数, 点の数)。成分が 1 つなら evaluate と同じ"""
    mus, sds = np.atleast_2d(mus), np.atleast_2d(sds)
    mu = mus.mean(axis=0)
    cdf = norm.cdf((y - mus) / sds).mean(axis=0)
    log_dens = logsumexp(norm.logpdf(y, mus, sds), axis=0) - np.log(len(mus))
    sd_mix = np.sqrt((sds ** 2 + mus ** 2).mean(axis=0) - mu ** 2)
    return {"R2": float(r2_score(y, mu)) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean((cdf >= 0.025) & (cdf <= 0.975))), "NLPD": float(-np.mean(log_dens)), "mean_sd": float(np.mean(sd_mix))}


def to_arrays(trace, cols):
    """サンプルの表 → {パラメータ: (チェーン数, 反復数) の配列}"""
    t = trace.sort_values(["chain", "draw"])
    n_chain = t["chain"].nunique()
    return {c: t[c].to_numpy().reshape(n_chain, -1) for c in cols}


def diag_summary(trace, cols):
    """ArviZ による収束診断の表（R-hat、ESS の bulk・tail）"""
    import arviz as az
    return az.summary(az.from_dict({"posterior": to_arrays(trace, cols)}), kind="diagnostics", round_to="none")


def thin_draws(trace, cols, n):
    """全チェーンのサンプルから、等間隔に n 個選んだパラメータの行列（行ごとに 1 サンプル）"""
    t = trace.sort_values(["chain", "draw"])
    a = t[cols].to_numpy()
    return a[np.linspace(0, len(a) - 1, min(n, len(a))).astype(int)]

## 2. データの読み込みと、部品の確認

手順3d の 2 つの版の分け方（`version_sets`）で、学習データとテストデータを作る。`SETS5B` で選んだ学習データごと・モデルごとに、MCMC を行う。

**部品の確認**（判定）：条件ごとにまとめた対数尤度と予測が、scikit-learn のショットごとの GPR（同じカーネルの値）と、1e-6 以内で一致する。

In [ ]:
# セル2：読み込みと、学習データ・テストデータの用意
D = load_all()
VS = {(mat, ver): version_sets(mat, ver, MAIN_DATA, TEMP_HOLDOUT) for ver in DATA_VERSIONS for mat in MATS}
TRAIN_DF, TESTS = {}, {}
for (mat, ver), vs in VS.items():
    for n in SETS5B[ver]:
        if n == "ALL":
            TRAIN_DF[(mat, ver, n)], TESTS[(mat, ver, n)] = vs["pool"], {}
        elif n in vs["splits"]:
            TRAIN_DF[(mat, ver, n)], TESTS[(mat, ver, n)] = vs["splits"][n]
        else:
            print(f"（{MATS[mat]}・{ver} 版に、分け方 {n} はない。飛ばす）")
MODEL_SPEC = {key: tuple(key.split("_")) for key in MCMC_MODELS}          # "B_1" → ("B", "1")
COMBOS = [(mat, ver, n, key) for mat in MATS for ver in DATA_VERSIONS for n in SETS5B[ver] if (mat, ver, n) in TRAIN_DF for key in MCMC_MODELS]
GD = {c: make_grouped(TRAIN_DF[c[:3]], *MODEL_SPEC[c[3]]) for c in COMBOS}       # 条件ごとにまとめた学習データ
for (mat, ver, n), tr in TRAIN_DF.items():
    print(f"{MATS[mat]}・{ver} 版・{n}：学習 {len(tr)} 行 {tr['cond'].nunique()} 条件、テスト " + ("、".join(f"{k} {len(t)} 行" for k, t in TESTS[(mat, ver, n)].items()) or "なし"))
print(f"モデル {MCMC_MODELS}、組の数 {len(COMBOS)}（材料 × 版 × 学習データ × モデル）")

BEST5 = pd.read_csv(os.path.join(STEP5_DIR, "best_E.csv"), encoding="utf-8-sig", dtype={"hv": str}) if os.path.exists(os.path.join(STEP5_DIR, "best_E.csv")) else None
print("手順5 の最良の E：" + ("読み込んだ" if BEST5 is not None else "結果がないので、E_START を使う"))


def e0_of(mat, ver, n, struct, hv):
    """手順5 のプロファイル尤度の最良の E（そのモデルの学習データのもの）。なければ E_START"""
    if struct == "C":
        return []
    if BEST5 is not None:
        sn = "S_V" if (n == "ALL" and ver == "5data") else n
        r = BEST5[(BEST5["material"] == MATS[mat]) & (BEST5["version"] == ver) & (BEST5["set"] == sn) & (BEST5["struct"] == struct) & (BEST5["hv"] == hv)]
        if len(r):
            return [float(r["Eh_best"].iloc[0]), float(r["Ev_best"].iloc[0])] if struct == "B" else [float(r["Ev_best"].iloc[0])]
    return list(E_START[struct])

In [ ]:
# セル3：部品の確認（条件ごとにまとめた尤度・予測が、scikit-learn のショットごとの GPR と一致するか）
mat1, ver1, n1 = min(((m, v, n) for (m, v, n) in TRAIN_DF if TESTS[(m, v, n)]), key=lambda k: len(TRAIN_DF[k]))
tr1, te1 = TRAIN_DF[(mat1, ver1, n1)], list(TESTS[(mat1, ver1, n1)].values())[0]
for key in MCMC_MODELS:
    st, hv = MODEL_SPEC[key]
    e0 = e0_of(mat1, ver1, n1, st, hv)
    v = {"struct": st, "hv": hv, "pcol": "ln_P32_cm", "Eh": e0[0] if st == "B" else 0.0, "Ev": e0[-1] if st != "C" else 0.0}
    f = fit5(v, tr1, 1)                                                  # scikit-learn のショットごとの GPR（やり直し 1 回）
    th = f["gp"].kernel_.theta
    gd1 = make_grouped(tr1, st, hv)
    l_sk, l_gr = float(f["gp"].log_marginal_likelihood_value_), grouped_lml(gd1, v, kernel5(st, hv), th)
    phi = np.r_[th, [v[nm] for nm in E_NAMES[st]]]
    mu_sk, sd_sk, _, _ = predict5(f, te1)
    mu_gr, sd_gr, _ = grouped_predict(gd1, st, hv, phi, te1)
    diff = max(abs(l_sk - l_gr), float(np.abs(mu_sk - mu_gr).max()), float(np.abs(sd_sk - sd_gr).max()))
    judge(diff < 1e-6, f"{key}：条件ごとにまとめた尤度・予測が、ショットごとの GPR と一致（{MATS[mat1]}・{ver1} 版・{n1}：対数尤度 {l_gr:.4f} と {l_sk:.4f}、最大の差 {diff:.1e}）")
c0 = COMBOS[0]
st0, hv0 = MODEL_SPEC[c0[3]]
k0 = kernel5(st0, hv0)
v0 = model_v(st0, hv0, np.r_[k0.theta, e0_of(c0[0], c0[1], c0[2], st0, hv0)])
t0 = time.time()
for _ in range(100):
    grouped_lml(GD[c0], v0, k0, k0.theta)
print(f"尤度 1 回あたり {(time.time() - t0) * 10:.2f} ミリ秒")

## 3. 最尤推定（比べる基準）

同じ尤度を、scipy の L-BFGS-B で最大化する（8 通りの出発点）。2 通りの最尤推定を作る。
- **MLE（E 固定）**：E を、手順5 のプロファイル尤度の最良値に固定して、カーネルだけを推定（手順5 の評価と同じ。C_1 は E がないので、同時と同じ）。
- **MLE（同時）**：E も動かして、カーネルと同時に推定。MCMC の初期値にもなる。

結果は `mle.csv` に保存する。

In [ ]:
# セル4：最尤推定（並列）
def mle_job(gd, struct, hv, e_prior, e0, seed):
    warnings.simplefilter("ignore")
    phi_f, l_f = fit_mle5(gd, struct, hv, e_prior, e0, True, 8, seed)
    phi_j, l_j = fit_mle5(gd, struct, hv, e_prior, e0, False, 8, seed)
    return phi_f, l_f, phi_j, l_j


t_all = time.time()
res = Parallel(n_jobs=N_JOBS)(delayed(mle_job)(GD[c], *MODEL_SPEC[c[3]], E_PRIOR, e0_of(c[0], c[1], c[2], *MODEL_SPEC[c[3]]), SEED + i) for i, c in enumerate(COMBOS))
MLE_FIX = {c: r[0] for c, r in zip(COMBOS, res)}
MLE_JOINT = {c: r[2] for c, r in zip(COMBOS, res)}
rows = []
for c, r in zip(COMBOS, res):
    names = theta_names5(*MODEL_SPEC[c[3]])
    for meth, phi, ll in [("E 固定", r[0], r[1]), ("同時", r[2], r[3])]:
        rows.append({"material": MATS[c[0]], "version": c[1], "split": c[2], "model": c[3], "method": meth, "logLik": ll, **dict(zip(names, phi))})
mle_df = pd.DataFrame(rows)
mle_df.to_csv(os.path.join(OUT_DIR, "mle.csv"), index=False, encoding="utf-8-sig")
print(f"最尤推定：{len(COMBOS)} 組、{(time.time() - t_all) / 60:.1f} 分")
print(mle_df[mle_df["model"].str.startswith("B")][["material", "version", "split", "model", "method", "logLik", "Eh", "Ev"]].round(2).to_string(index=False))

## 4. MCMC によるサンプリング

- 4 本のチェーンの初期値は、MLE（同時）に、ノイズ以外のカーネルの対数パラメータへ N(0, 0.3²)、E へ N(0, 0.5²) [kJ/mol] のずれを加えたもの。
- 提案分布の最初の大きさは、対数パラメータ 0.05、E 0.5 kJ/mol（調整期間に、足跡から学び直す）。
- 本番のサンプルは `results/step5b_mcmc/posterior_<版>_<学習>_<材料>_<モデル>.csv` に、チェーンごとの採択率などは `chain_info.csv` に保存する。

In [ ]:
# セル5：サンプリング（保存済みのサンプルがあれば読み込む）
def post_path(c):
    mat, ver, n, key = c
    return os.path.join(OUT_DIR, f"posterior_{ver}_{n}_{mat}_{key}.csv")


def starts_seeds(i, c):
    """チェーンの初期値（MLE（同時）にゆらぎを足す）とシード"""
    st, hv = MODEL_SPEC[c[3]]
    names = theta_names5(st, hv)
    _, lo, hi = make_logpost(GD[c], st, hv, E_PRIOR)
    rng = np.random.default_rng(SEED + 100 * i)
    scale = np.array([0.0 if nm == "log_noise" else (0.5 if nm in E_NAMES[st] else 0.3) for nm in names])
    starts = [np.clip(MLE_JOINT[c] + rng.normal(0, 1, len(names)) * scale, lo + 1e-6, hi - 1e-6) for _ in range(CHAINS)]
    return starts, [SEED + 1000 * i + k for k in range(CHAINS)]


info_path = os.path.join(OUT_DIR, "chain_info.csv")
chain_info = pd.read_csv(info_path) if os.path.exists(info_path) else pd.DataFrame()
traces = {}
for c in COMBOS:
    if USE_SAVED_SAMPLES and os.path.exists(post_path(c)):
        saved = pd.read_csv(post_path(c))
        per_chain = saved.groupby("chain").size()
        if len(per_chain) == CHAINS and (per_chain >= N_SAMP).all():
            traces[c] = saved
            print(f"{MATS[c[0]]}・{c[1]}・{c[2]}・{c[3]}: 保存済みのサンプルを読み込み（1チェーン {int(per_chain.iloc[0])} 回）")
pending, n_samp = [c for c in COMBOS if c not in traces], N_SAMP
t_all = time.time()
while pending:
    tasks = []
    for c in pending:
        st, hv = MODEL_SPEC[c[3]]
        starts, seeds = starts_seeds(COMBOS.index(c), c)
        init_sd = np.array([0.5 if nm in E_NAMES[st] else 0.05 for nm in theta_names5(st, hv)])
        tasks += [(c, k, st, hv, starts[k], init_sd, seeds[k]) for k in range(CHAINS)]
    t0 = time.time()
    outs = Parallel(n_jobs=N_JOBS)(delayed(run_chain5)(GD[t[0]], t[2], t[3], E_PRIOR, t[4], t[5], N_WARM, n_samp, t[6]) for t in tasks)
    print(f"本番 {n_samp:,}回 × {CHAINS}チェーン × {len(pending)} 組：{(time.time() - t0) / 60:.1f} 分")
    still = []
    for c in pending:
        mat, ver, n, key = c
        res_c = [o for (t, o) in zip(tasks, outs) if t[0] == c]
        names = theta_names5(*MODEL_SPEC[key])
        trace = pd.concat([pd.DataFrame({"chain": k, "draw": np.arange(n_samp), **{nm: r["draws"][:, j] for j, nm in enumerate(names)},
                                         "lp": r["lp"], "accepted": r["accepted"].astype(int)}) for k, r in enumerate(res_c)], ignore_index=True)
        dg = diag_summary(trace, names)
        rhat, ess = float(dg["r_hat"].max()), float(dg["ess_bulk"].min())
        print(f"  {MATS[mat]}・{ver}・{n}・{key}: 最大 R-hat {rhat:.3f}、最小 ESS(bulk) {ess:.0f}、採択率 " + ", ".join(f"{r['acc_samp']:.2f}" for r in res_c))
        if (rhat > 1.01 or ess < 400) and n_samp < N_SAMP_MAX:
            still.append(c)                                           # 収束していない（または有効サンプル数が少ない）ので、本番を増やしてやり直す
            continue
        trace.to_csv(post_path(c), index=False, float_format="%.7g")
        traces[c] = pd.read_csv(post_path(c))                         # 保存した値（7桁）で以降を計算する
        rows_ = pd.DataFrame([{"material": MATS[mat], "version": ver, "split": n, "model": key, "chain": k, "acc_warm": r["acc_warm"], "acc_samp": r["acc_samp"],
                               "lambda": r["lam"], "chain_seconds": r["seconds"], "wall_seconds": max(x["seconds"] for x in res_c)} for k, r in enumerate(res_c)])
        if len(chain_info):
            chain_info = chain_info[~((chain_info["material"] == MATS[mat]) & (chain_info["version"] == ver) & (chain_info["split"] == n) & (chain_info["model"] == key))]
        chain_info = pd.concat([chain_info, rows_], ignore_index=True)
        chain_info.to_csv(info_path, index=False)
    pending, n_samp = still, min(int(n_samp * 2.5), N_SAMP_MAX)
RUN_INFO = {c: {"chains": int(t["chain"].nunique()), "n_warm": N_WARM, "n_samp_per_chain": int(len(t) // t["chain"].nunique()), "draws_total": int(len(t))} for c, t in traces.items()}
print(f"\n合計 {(time.time() - t_all) / 60:.1f} 分")
print(pd.DataFrame([{"material": MATS[c[0]], "version": c[1], "split": c[2], "model": c[3], **i} for c, i in RUN_INFO.items()]).to_string(index=False))

## 5. 収束診断と事後分布

- R-hat（1.01 以下が目安）と有効サンプル数 ESS（bulk・tail。400 以上が目安）は ArviZ で計算する。
- 事後分布の要約は、振幅（√振幅²）・長さスケール・σ0・ノイズ分散の元の単位と、E［kJ/mol］で示す（長さスケールは標準化後の入力に対する値）。
- 判定：①全組で収束（R-hat ≤ 1.01、ESS ≥ 400）。②事後分布の 95% 区間が、事前分布の範囲の端（下限・上限の 5% 以内）に届いたパラメータ（届いたら、データだけでは決まらない）。③E の事後分布が、E の事前分布の範囲の端に届いていない。④E の事後分布の 95% 区間に、手順5 のプロファイル尤度の最良値が入る。結果は `posterior_summary.csv`・`sampler_stats.csv`。

In [ ]:
# セル6：収束診断と事後分布の要約
def to_orig(name, x):
    """対数パラメータ → 元の単位（振幅² は振幅にする）。E はそのまま。(ラベル, 値)"""
    if name in ("Eh", "Ev"):
        return name, np.asarray(x, float)
    x = np.exp(np.asarray(x, float))
    if name.startswith("log_c_lin"):
        return name.replace("log_c_lin", "lin_amp"), np.sqrt(x)
    if name.startswith("log_c"):
        return name.replace("log_c", "amp"), np.sqrt(x)
    return name.replace("log_", ""), x


rows = []
for c, trace in traces.items():
    mat, ver, n, key = c
    st, hv = MODEL_SPEC[key]
    names = theta_names5(st, hv)
    dg = diag_summary(trace, names)
    _, lo, hi = make_logpost(GD[c], st, hv, E_PRIOR)
    e0 = e0_of(mat, ver, n, st, hv)
    for j, nm in enumerate(names):
        lab, v = to_orig(nm, trace[nm].to_numpy())
        _, lo_b = to_orig(nm, lo[j])
        _, hi_b = to_orig(nm, hi[j])
        _, m_f = to_orig(nm, MLE_FIX[c][j])
        _, m_j = to_orig(nm, MLE_JOINT[c][j])
        q = np.percentile(v, [2.5, 50, 97.5])
        span = float(hi_b - lo_b) if nm in E_NAMES[st] else None
        at_bound = bool(q[0] < lo_b + 0.05 * span or q[2] > hi_b - 0.05 * span) if span else bool(q[0] < lo_b * 1.05 or q[2] > hi_b * 0.95)
        prof = e0[E_NAMES[st].index(nm)] if nm in E_NAMES[st] and e0 else np.nan
        rows.append({"material": MATS[mat], "version": ver, "split": n, "model": key, **RUN_INFO[c], "param": lab, "MLE_E_fixed": float(m_f), "MLE_joint": float(m_j), "profile_best_E": prof,
                     "post_mean": float(v.mean()), "post_sd": float(v.std(ddof=1)), "q2.5": q[0], "median": q[1], "q97.5": q[2], "r_hat": float(dg.loc[nm, "r_hat"]),
                     "ess_bulk": float(dg.loc[nm, "ess_bulk"]), "ess_tail": float(dg.loc[nm, "ess_tail"]), "post_at_bound": at_bound})
post_df = pd.DataFrame(rows)
post_df.to_csv(os.path.join(OUT_DIR, "posterior_summary.csv"), index=False, encoding="utf-8-sig")
show_ = post_df[post_df["split"].isin(["S_V", "ALL"])]
print(show_[["material", "version", "split", "model", "param", "MLE_joint", "median", "q2.5", "q97.5", "r_hat", "ess_bulk", "post_at_bound"]].to_string(index=False, float_format=lambda x: f"{x:.4g}"))
stat_rows = []
for c in traces:
    ci = chain_info[(chain_info["material"] == MATS[c[0]]) & (chain_info["version"] == c[1]) & (chain_info["split"] == c[2]) & (chain_info["model"] == c[3])]
    p = post_df[(post_df["material"] == MATS[c[0]]) & (post_df["version"] == c[1]) & (post_df["split"] == c[2]) & (post_df["model"] == c[3])]
    stat_rows.append({"material": MATS[c[0]], "version": c[1], "split": c[2], "model": c[3], **RUN_INFO[c], "acc_samp_min": ci["acc_samp"].min(), "acc_samp_max": ci["acc_samp"].max(),
                      "sampling_min": ci["wall_seconds"].iloc[0] / 60, "max_r_hat": p["r_hat"].max(), "min_ess_bulk": p["ess_bulk"].min(), "min_ess_tail": p["ess_tail"].min(),
                      "n_post_at_bound": int(p["post_at_bound"].sum()), "params_at_bound": ", ".join(p[p["post_at_bound"]]["param"])})
stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(os.path.join(OUT_DIR, "sampler_stats.csv"), index=False, encoding="utf-8-sig")
print("\n【サンプラーの統計】")
print(stat_df.round(3).to_string(index=False))
n_bad = int(((stat_df["max_r_hat"] > 1.01) | (stat_df["min_ess_bulk"] < 400)).sum())
judge(n_bad == 0, f"全 {len(stat_df)} 組で R-hat ≤ 1.01、ESS(bulk) ≥ 400（満たさないのは {n_bad} 組。満たさないなら、N_SAMP_MAX を増やして再実行）")
for key in MCMC_MODELS:
    s = stat_df[stat_df["model"] == key]
    judge(int((s["n_post_at_bound"] > 0).sum()) == 0, f"{key}：事後分布の 95% 区間が、事前分布の範囲の端に届いた組は {int((s['n_post_at_bound'] > 0).sum())} / {len(s)}（届いたパラメータ: {sorted(set(', '.join(s['params_at_bound']).split(', ')) - {''})}。届いたら、そのパラメータは、データだけでは決まらない）")
pe = post_df[post_df["param"].isin(["Eh", "Ev"])]
if len(pe):
    judge(not bool(pe["post_at_bound"].any()), f"E の事後分布が、E の事前分布の範囲の端（5% 以内）に届いていない（届いた: {int(pe['post_at_bound'].sum())} / {len(pe)}）")
    inside = (pe["profile_best_E"] >= pe["q2.5"]) & (pe["profile_best_E"] <= pe["q97.5"])
    judge(bool(inside.all()), f"E の事後分布の 95% 区間に、手順5 のプロファイル尤度の最良値が入る（入った: {int(inside.sum())} / {len(pe)}。外れたもの: {sorted(set(pe[~inside]['model'] + ' ' + pe[~inside]['split'] + ' ' + pe[~inside]['param']))}）")

## 6. 事後分布の図

- 図1：B_1 の E_h と E_v の事後分布（散布図。手順5 の最良値と MLE を重ねる）。斜めの帯になるか。
- 図2：差 E_v − E_h の事後分布（ヒストグラム）。
- 図3：ハイパーパラメータの事後分布（ヒストグラム。log10。赤の破線が MLE（同時）、灰色の点線が事前分布の範囲の端）。版ごとの主な学習データ（5data 版は S_V、Alldata 版は ALL）のもの。
- 図4：トレース（チェーンごとの推移）。同じく主な学習データのもの。

すべて `save_fig` で保存する（通常の PNG と、`highres/` の 600 dpi）。

In [ ]:
# セル7：図の共通の設定、図1・図2（E の事後分布）
CHAIN_COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
T_COL = {190: "tab:blue", 210: "tab:green", 230: "tab:red"}
H_LEVELS = [0.5, 1.0, 1.5, 2.0]
H_COL = {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"}


def main_set(ver):
    n = "S_V" if ver == "5data" else "ALL"
    return n if n in SETS5B[ver] else SETS5B[ver][-1]


def thinned(c, n=N_PRED_DRAWS):
    names = theta_names5(*MODEL_SPEC[c[3]])
    return thin_draws(traces[c], names, n)


bkeys = [k for k in MCMC_MODELS if MODEL_SPEC[k][0] == "B"]
if bkeys:
    key = bkeys[0]
    for ver in DATA_VERSIONS:
        sets_ = [n for n in SETS5B[ver] if (list(MATS)[0], ver, n, key) in traces]
        fig, axes = plt.subplots(len(MATS), len(sets_), figsize=(5.4 * len(sets_), 4.8 * len(MATS)), squeeze=False)
        fig2, axes2 = plt.subplots(len(MATS), len(sets_), figsize=(5.4 * len(sets_), 3.6 * len(MATS)), squeeze=False)
        for i, mat in enumerate(MATS):
            for j, n in enumerate(sets_):
                c = (mat, ver, n, key)
                t = traces[c]
                ax, ax2 = axes[i, j], axes2[i, j]
                ax.scatter(t["Eh"][::10], t["Ev"][::10], s=3, alpha=0.25, color="tab:purple", label="事後サンプル（間引き）")
                e0 = e0_of(mat, ver, n, "B", MODEL_SPEC[key][1])
                ax.plot([e0[0]], [e0[1]], "r*", ms=14, mec="k", label=f"手順5 の最良値（{e0[0]:.0f}, {e0[1]:.0f}）")
                mj = MLE_JOINT[c]
                ax.plot([mj[-2]], [mj[-1]], "kx", ms=9, mew=2, label="MLE（同時）")
                lim = [min(t["Eh"].min(), t["Ev"].min()) - 1, max(t["Eh"].max(), t["Ev"].max()) + 1]
                ax.plot(lim, lim, "k--", lw=0.8, label="E_h = E_v")
                ax.set_xlabel("横方向 E_h [kJ/mol]")
                ax.set_ylabel("縦方向 E_v [kJ/mol]")
                ax.set_title(f"{MATS[mat]}・{ver} 版・{n} の学習", fontsize=10)
                ax.grid(alpha=0.3)
                if i == 0 and j == 0:
                    ax.legend(fontsize=7, loc="upper left")
                d = (t["Ev"] - t["Eh"]).to_numpy()
                ax2.hist(d, bins=40, color="tab:purple", alpha=0.7)
                ax2.axvline(e0[1] - e0[0], color="r", lw=2, label=f"手順5 の最良値（{e0[1] - e0[0]:.0f}）")
                q = np.percentile(d, [2.5, 97.5])
                ax2.axvspan(q[0], q[1], color="gray", alpha=0.15, label=f"95% 区間（{q[0]:.1f}〜{q[1]:.1f}）")
                ax2.set_xlabel("E_v − E_h [kJ/mol]")
                ax2.set_title(f"{MATS[mat]}・{ver} 版・{n}", fontsize=10)
                ax2.legend(fontsize=7)
        fig.suptitle(f"{key}：E_h と E_v の事後分布（{ver} 版）。斜めの帯 ＝ 差だけがよく決まる", fontsize=11)
        fig.tight_layout()
        save_fig(fig, OUT_DIR, f"fig1_E_posterior_{ver}")
        plt.show()
        fig2.suptitle(f"{key}：差 E_v − E_h の事後分布（{ver} 版）", fontsize=11)
        fig2.tight_layout()
        save_fig(fig2, OUT_DIR, f"fig2_E_diff_{ver}")
        plt.show()

In [ ]:
# セル8：図3 ハイパーパラメータの事後分布（版ごとの主な学習データ。材料×モデルごとに 1 枚）
INPUT_JA = {"u": "u（ln γ̇。B は横シフト後）", "H": "ln H", "T": "Tset", "w": "w（ln Fo）", "p": "ln P3_2"}


def param_ja(nm):
    if nm in ("Eh", "Ev"):
        return {"Eh": "横方向のずらし量 E_h", "Ev": "縦方向のずらし量 E_v"}[nm] + "（kJ/mol）"
    base = {"log_c": "振幅²", "log_noise": "ノイズ分散", "log_sigma0": "線形項の σ0", "log_c_lin": "線形項の振幅²"}
    if nm in base:
        return base[nm] + "（log10）"
    if nm.startswith("log_ls_"):
        rest = nm[len("log_ls_"):]
        return "長さスケール：" + INPUT_JA.get(rest.split("_")[-1], rest) + "（log10）"
    return nm


for ver in DATA_VERSIONS:
    for mat in MATS:
        for key in MCMC_MODELS:
            c = (mat, ver, main_set(ver), key)
            if c not in traces:
                continue
            st, hv = MODEL_SPEC[key]
            names = theta_names5(st, hv)
            _, lo, hi = make_logpost(GD[c], st, hv, E_PRIOR)
            ncol = 4
            nrow = (len(names) + ncol - 1) // ncol
            fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 2.9 * nrow), squeeze=False)
            for j, nm in enumerate(names):
                ax = axes[j // ncol, j % ncol]
                if nm in E_NAMES[st]:
                    v, mle, b0, b1 = traces[c][nm].to_numpy(), MLE_JOINT[c][j], lo[j], hi[j]
                else:
                    v, mle, b0, b1 = traces[c][nm].to_numpy() / np.log(10), MLE_JOINT[c][j] / np.log(10), lo[j] / np.log(10), hi[j] / np.log(10)
                ax.set_title(param_ja(nm), fontsize=9)
                ax.hist(v, bins=50, color="tab:blue", alpha=0.7, density=True)
                ax.axvline(mle, color="r", ls="--", lw=1.5, label="MLE（同時）")
                for b in (b0, b1):
                    if v.min() - 0.1 * (np.ptp(v) + 1e-9) <= b <= v.max() + 0.1 * (np.ptp(v) + 1e-9) or abs(b - np.median(v)) < 3 * (np.ptp(v) + 1e-9):
                        ax.axvline(b, color="gray", ls=":", lw=1.5)
                ax.tick_params(labelsize=7)
            for j in range(len(names), nrow * ncol):
                axes[j // ncol, j % ncol].axis("off")
            axes[0, 0].legend(fontsize=7)
            fig.suptitle(f"{MATS[mat]}・{ver} 版・{main_set(ver)}・{key}：ハイパーパラメータの事後分布。赤の破線：MLE（同時）、灰色の点線：事前分布の範囲の端（範囲内にあるとき）", fontsize=10)
            fig.tight_layout(rect=(0, 0, 1, 0.96))
            save_fig(fig, OUT_DIR, f"fig3_posterior_{ver}_{mat}_{key}")
            plt.show()

# 図4 トレース（版ごとの主な学習データ）
for ver in DATA_VERSIONS:
    for mat in MATS:
        for key in MCMC_MODELS:
            c = (mat, ver, main_set(ver), key)
            if c not in traces:
                continue
            names = theta_names5(*MODEL_SPEC[key])
            fig, axes = plt.subplots(len(names), 1, figsize=(11, 1.5 * len(names)), sharex=True, squeeze=False)
            for j, nm in enumerate(names):
                ax = axes[j, 0]
                for k in range(CHAINS):
                    tk = traces[c][traces[c]["chain"] == k]
                    ax.plot(tk["draw"], tk[nm] if nm in E_NAMES[MODEL_SPEC[key][0]] else tk[nm] / np.log(10), lw=0.4, color=CHAIN_COLORS[k % 4], alpha=0.8)
                ax.set_ylabel(nm, fontsize=7)
                ax.tick_params(labelsize=7)
            axes[-1, 0].set_xlabel("本番の反復")
            fig.suptitle(f"{MATS[mat]}・{ver} 版・{main_set(ver)}・{key}：トレース（色がチェーン。E 以外は log10）", fontsize=10)
            fig.tight_layout(rect=(0, 0, 1, 0.97))
            save_fig(fig, OUT_DIR, f"fig4_trace_{ver}_{mat}_{key}")
            plt.show()

## 7. 予測の比較（最尤推定と MCMC）

学習データごとのテストを、3 通りで予測する。
- **MLE（E 固定）**：手順5 と同じ（E は最良値に固定。カーネルは最尤推定）。
- **MLE（同時）**：E もカーネルと同時に最尤推定。
- **MCMC**：事後サンプル `N_PRED_DRAWS` 個の予測分布を、等しい重みで混ぜた混合分布（ハイパーパラメータと E の不確かさを含む）。

指標は RMSE・R²・被覆率（95% 予測区間）・NLPD・予測標準偏差の平均（区間の広さの目安）。結果は `metrics.csv`。テストのない学習データ（ALL）は、評価しない。

判定：①MCMC の被覆率が 0.90〜0.99。②MCMC の RMSE が MLE（E 固定）の 1.05 倍以内。③MCMC の NLPD が MLE（E 固定）より 0.05 を超えて悪くない。④不確かさを含めた分、MCMC の区間（平均の予測標準偏差）が、MLE より広い（ただし 1.5 倍以内）。

In [ ]:
# セル9：予測の指標
mrows = []
for c in COMBOS:
    mat, ver, n, key = c
    if not TESTS[(mat, ver, n)]:
        continue
    st, hv = MODEL_SPEC[key]
    th_draws = thinned(c)
    for tn, te in TESTS[(mat, ver, n)].items():
        y = te["ln_eta"].to_numpy()
        preds = {}
        for meth, phi in [("MLE（E 固定）", MLE_FIX[c]), ("MLE（同時）", MLE_JOINT[c])]:
            mu, sd, _ = grouped_predict(GD[c], st, hv, phi, te)
            preds[meth] = (mu[None], sd[None])
        mus, sds, _, _, _ = predict_mixture(GD[c], st, hv, th_draws, te)
        preds["MCMC"] = (mus, sds)
        subs = {"全体": np.ones(len(te), bool), **{g: (te["group"] == g).to_numpy() for g in te["group"].unique() if g != "全体"}}
        for meth, (m_, s_) in preds.items():
            for sub, mask in subs.items():
                if mask.sum() < 2:
                    continue
                ev = evaluate_mixture(y[mask], m_[:, mask], s_[:, mask])
                mrows.append({"material": MATS[mat], "version": ver, "split": tn, "train_set": n, "subset": sub, "model": key, "method": meth, "n_test": int(mask.sum()), **ev})
met_df = pd.DataFrame(mrows)
met_df.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding="utf-8-sig")
METHODS = ["MLE（E 固定）", "MLE（同時）", "MCMC"]
PANELS = [("S_V", "低速側"), ("S_V", "高速側"), ("S_T", "全体"), ("S_T3", "全体")]
for mat in MATS:
    for ver in DATA_VERSIONS:
        t = met_df[(met_df["material"] == MATS[mat]) & (met_df["version"] == ver)]
        if t.empty:
            continue
        print(f"\n【{MATS[mat]}・{ver} 版】RMSE／被覆率／NLPD／平均の予測標準偏差")
        t = t.assign(panel=t["split"] + " " + t["subset"])
        t = t.assign(v=t["RMSE"].map("{:.3f}".format) + "／" + t["coverage95"].map("{:.2f}".format) + "／" + t["NLPD"].map("{:.2f}".format) + "／" + t["mean_sd"].map("{:.3f}".format))
        print(t.pivot_table(index=["model", "method"], columns="panel", values="v", aggfunc="first").reindex(index=[(k, m) for k in MCMC_MODELS for m in METHODS]).to_string())


def mm(mat, ver, split, sub, key, meth, col):
    r = met_df[(met_df["material"] == MATS[mat]) & (met_df["version"] == ver) & (met_df["split"] == split) & (met_df["subset"] == sub) & (met_df["model"] == key) & (met_df["method"] == meth)]
    return float(r[col].iloc[0]) if len(r) else np.nan


for mat in MATS:
    for ver in DATA_VERSIONS:
        for key in MCMC_MODELS:
            t = met_df[(met_df["material"] == MATS[mat]) & (met_df["version"] == ver) & (met_df["model"] == key) & (met_df["subset"] == "全体")]
            if t.empty:
                continue
            a = t[t["method"] == "MCMC"].set_index("split")
            b = t[t["method"] == "MLE（E 固定）"].set_index("split")
            tag = f"{MATS[mat]}・{ver} 版・{key}"
            judge(bool(a["coverage95"].between(0.90, 0.99).all()), f"{tag}：MCMC の被覆率が 0.90〜0.99（{', '.join(f'{s} {v:.2f}' for s, v in a['coverage95'].items())}）")
            judge(bool((a["RMSE"] <= 1.05 * b["RMSE"]).all()), f"{tag}：MCMC の RMSE が、MLE（E 固定）の 1.05 倍以内（{', '.join(f'{s} {a.loc[s, 'RMSE']:.3f} 対 {b.loc[s, 'RMSE']:.3f}' for s in a.index)}）")
            judge(bool((a["NLPD"] <= b["NLPD"] + 0.05).all()), f"{tag}：MCMC の NLPD が、MLE（E 固定）より 0.05 を超えて悪くない（{', '.join(f'{s} {a.loc[s, 'NLPD']:.2f} 対 {b.loc[s, 'NLPD']:.2f}' for s in a.index)}）")
            ratio = a["mean_sd"] / b["mean_sd"]
            judge(bool(((ratio >= 1.0 - 1e-9) & (ratio <= 1.5)).all()), f"{tag}：MCMC の区間（平均の予測標準偏差）が MLE より広い（1.0〜1.5 倍。{', '.join(f'{s} {r:.2f} 倍' for s, r in ratio.items())}）")

In [ ]:
# セル10：図5 予測指標の比較（MLE と MCMC）
if len(met_df):
    colors = {"MLE（E 固定）": "tab:blue", "MLE（同時）": "tab:green", "MCMC": "tab:red"}
    for ver in DATA_VERSIONS:
        t = met_df[met_df["version"] == ver]
        panels = [(s, g) for s, g in PANELS if ((t["split"] == s) & (t["subset"] == g)).any()]
        if not panels:
            continue
        fig, axes = plt.subplots(3, len(MATS), figsize=(8.5 * len(MATS), 12), squeeze=False)
        for j, mat in enumerate(MATS):
            for i, (metric, ylab) in enumerate([("RMSE", r"RMSE（$\ln\eta$）"), ("coverage95", "95% 区間の被覆率"), ("NLPD", "NLPD（小さいほど良い）")]):
                ax = axes[i, j]
                groups = [(key, p) for key in MCMC_MODELS for p in panels]
                w = 0.8 / len(METHODS)
                for k, meth in enumerate(METHODS):
                    ax.bar(np.arange(len(groups)) + (k - 1) * w, [mm(mat, ver, p[0], p[1], key, meth, metric) for key, p in groups], w, color=colors[meth], label=meth)
                ax.set_xticks(np.arange(len(groups)))
                ax.set_xticklabels([f"{key}\n{p[0]} {p[1]}" for key, p in groups], fontsize=7)
                ax.set_title(f"{MATS[mat]}・{ver} 版：{ylab}", fontsize=10)
                ax.grid(alpha=0.3, axis="y")
                if metric == "coverage95":
                    ax.axhline(0.95, color="k", lw=0.8, ls="--")
                    ax.set_ylim(0.6, 1.02)
                if i == 0 and j == 0:
                    ax.legend(fontsize=8)
        fig.tight_layout()
        save_fig(fig, OUT_DIR, f"fig5_metrics_{ver}")
        plt.show()

## 8. 設定温度の効果の不確かさ

最終モデルでなく、各学習データのモデルで、5data＋3data の 210 ℃ のショットについて、ほかの入力はそのまま、Tset だけを 190 ℃・230 ℃ にして予測した粘度の比（190 ℃ ÷ 230 ℃）を、厚みごとに中央値でまとめる。MCMC は、事後サンプルごとに計算して、95% 区間を出す。実測の比（同じ γ̇、厚みごと）と比べる。結果は `tset_effect.csv`・`observed_ratio.csv`。

判定：B 系のモデルで、実測の比が、MCMC の 95% 区間に入る厚みが 3 つ以上（4 つのうち）。

In [ ]:
# セル11：設定温度の効果
def observed_ratio(mat):
    d = pool_of(mat, MAIN_DATA)
    d = d[~d["is_sens"]]
    cm = d.groupby(["Tset", "H", "v"]).agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean")).reset_index()
    rows = []
    for h in H_LEVELS:
        a, c_ = cm[(cm["H"] == h) & (cm["Tset"] == 190)], cm[(cm["H"] == h) & (cm["Tset"] == 230)].sort_values("lg")
        ins = ((a["lg"] >= c_["lg"].min()) & (a["lg"] <= c_["lg"].max())).to_numpy()
        r = np.exp(a["le"].to_numpy()[ins] - np.interp(a["lg"].to_numpy()[ins], c_["lg"], c_["le"]))
        rows.append({"material": MATS[mat], "H": h, "observed": float(np.median(r)), "n": int(ins.sum())})
    return pd.DataFrame(rows)


obs_df = pd.concat([observed_ratio(m) for m in MATS], ignore_index=True)
obs_df.to_csv(os.path.join(OUT_DIR, "observed_ratio.csv"), index=False, encoding="utf-8-sig")
trows = []
for c in COMBOS:
    mat, ver, n, key = c
    st, hv = MODEL_SPEC[key]
    base = pool_of(mat, MAIN_DATA)
    base = base[(base["Tset"] == 210) & ~base["is_sens"]]
    lo_, hi_ = base.copy(), base.copy()
    lo_["Tset"], hi_["Tset"] = 190.0, 230.0

    def ratio_by_h(phis):
        m_lo = predict_mixture(GD[c], st, hv, phis, lo_)[0]
        m_hi = predict_mixture(GD[c], st, hv, phis, hi_)[0]
        r = np.exp(m_lo - m_hi)                                                    # (サンプル数, 行数)
        return np.array([np.median(r[:, (base["H"] == h).to_numpy()], axis=1) for h in H_LEVELS])    # (厚み, サンプル数)

    for meth, phis in [("MLE（E 固定）", MLE_FIX[c][None]), ("MLE（同時）", MLE_JOINT[c][None]), ("MCMC", thinned(c))]:
        rb = ratio_by_h(phis)
        for i, h in enumerate(H_LEVELS):
            q = np.percentile(rb[i], [2.5, 50, 97.5])
            trows.append({"material": MATS[mat], "version": ver, "split": n, "model": key, "method": meth, "H": h, "ratio_median": q[1], "ratio_q2.5": q[0], "ratio_q97.5": q[2]})
eff_df = pd.DataFrame(trows)
eff_df.to_csv(os.path.join(OUT_DIR, "tset_effect.csv"), index=False, encoding="utf-8-sig")
print("実測の 190/230 ℃ の粘度比（同じ γ̇、厚みごと）")
print(obs_df.pivot(index="material", columns="H", values="observed").round(3).to_string())
for ver in DATA_VERSIONS:
    for mat in MATS:
        n = main_set(ver)
        t = eff_df[(eff_df["material"] == MATS[mat]) & (eff_df["version"] == ver) & (eff_df["split"] == n)]
        if t.empty:
            continue
        print(f"\n【{MATS[mat]}・{ver} 版・{n} の学習】中央値（95% 区間）")
        t = t.assign(v=t["ratio_median"].map("{:.3f}".format) + "（" + t["ratio_q2.5"].map("{:.3f}".format) + "〜" + t["ratio_q97.5"].map("{:.3f}".format) + "）")
        print(t.pivot_table(index=["model", "method"], columns="H", values="v", aggfunc="first").to_string())
        for key in [k for k in MCMC_MODELS if MODEL_SPEC[k][0] == "B"]:
            tt = t[(t["model"] == key) & (t["method"] == "MCMC")].sort_values("H")
            o = obs_df[obs_df["material"] == MATS[mat]].sort_values("H")["observed"].to_numpy()
            inside = (o >= tt["ratio_q2.5"].to_numpy()) & (o <= tt["ratio_q97.5"].to_numpy())
            judge(int(inside.sum()) >= 3, f"{MATS[mat]}・{ver} 版・{key}：実測の 190/230 ℃ の粘度比が、MCMC の 95% 区間に入る厚みが {int(inside.sum())} / 4（3 以上なら OK）")

In [ ]:
# セル12：図6 設定温度の効果（区間と実測）
for ver in DATA_VERSIONS:
    fig, axes = plt.subplots(1, len(MATS), figsize=(8.5 * len(MATS), 5.4), sharey=True, squeeze=False)
    for ax, mat in zip(axes[0], MATS):
        o = obs_df[obs_df["material"] == MATS[mat]].sort_values("H")
        ax.plot(o["H"], o["observed"], "k-o", lw=2.8, ms=9, label="実測（同じ γ̇）", zorder=10)
        n = main_set(ver)
        for k, key in enumerate(MCMC_MODELS):
            t = eff_df[(eff_df["material"] == MATS[mat]) & (eff_df["version"] == ver) & (eff_df["split"] == n) & (eff_df["model"] == key)]
            if t.empty:
                continue
            col = ["tab:purple", "tab:gray", "tab:blue", "tab:orange"][k % 4]
            m = t[t["method"] == "MCMC"].sort_values("H")
            ax.fill_between(m["H"], m["ratio_q2.5"], m["ratio_q97.5"], color=col, alpha=0.25, label=f"{key}：MCMC の 95% 区間")
            ax.plot(m["H"], m["ratio_median"], "-", color=col, lw=1.6, label=f"{key}：MCMC の中央値")
            ml = t[t["method"] == "MLE（E 固定）"].sort_values("H")
            ax.plot(ml["H"], ml["ratio_median"], "--", color=col, lw=1.2, label=f"{key}：MLE（E 固定）")
        ax.axhline(1.0, color="gray", lw=0.8)
        ax.set_xlabel("厚み [mm]")
        ax.set_title(f"{MATS[mat]}・{ver} 版（{main_set(ver)} の学習）：190 ℃ と 230 ℃ の粘度の比", fontsize=10)
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("粘度の比（190 ℃ ÷ 230 ℃）")
    axes[0, 0].legend(fontsize=7)
    fig.tight_layout()
    save_fig(fig, OUT_DIR, f"fig6_tset_effect_{ver}")
    plt.show()

In [ ]:
# セル13：図7 設定温度ごとの予測曲線（MCMC の混合分布。行が厚み、列がモデル。版ごとの主な学習データ）
for ver in DATA_VERSIONS:
    for mat in MATS:
        n = main_set(ver)
        if (mat, ver, n, MCMC_MODELS[0]) not in traces:
            continue
        pool = pool_of(mat, MAIN_DATA) if ver == "5data" else D[(mat, "Alldata", MAIN_DATA)]
        tconds = set(TRAIN_DF[(mat, ver, n)]["cond"])
        fig, axes = plt.subplots(4, len(MCMC_MODELS), figsize=(5.2 * len(MCMC_MODELS), 14), sharex="row", sharey="row", squeeze=False)
        for j, key in enumerate(MCMC_MODELS):
            c = (mat, ver, n, key)
            st, hv = MODEL_SPEC[key]
            phis = thinned(c, 60)
            for i, h in enumerate(H_LEVELS):
                ax = axes[i, j]
                for t, col in T_COL.items():
                    sub = pool[(pool["H"] == h) & (pool["Tset"] == t)]
                    if sub.empty:
                        continue
                    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    lg = np.linspace(sub["ln_gd"].min(), sub["ln_gd"].max(), 100)
                    _, _, mu, sd, _ = predict_mixture(GD[c], st, hv, phis, hr_inputs(sub, lg, h, t))
                    ax.plot(np.exp(lg), np.exp(mu), color=col, lw=1.4)
                    ax.fill_between(np.exp(lg), np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd), color=col, alpha=0.15)
                    is_tr = cm["cond"].isin(tconds).to_numpy()
                    ax.scatter(np.exp(cm["lg"][is_tr]), np.exp(cm["le"][is_tr]), s=14, color=col, alpha=0.8, zorder=3)
                    ax.scatter(np.exp(cm["lg"][~is_tr]), np.exp(cm["le"][~is_tr]), s=26, facecolors="none", edgecolors=col, marker="s", linewidths=1.2, zorder=3)
                    se = cm["sens"].to_numpy().astype(bool)
                    ax.scatter(np.exp(cm["lg"][se]), np.exp(cm["le"][se]), s=45, color="k", marker="x", zorder=4)
                ax.set_xscale("log")
                ax.set_yscale("log")
                ax.xaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x_, _: f"{x_:g}"))
                ax.xaxis.set_minor_formatter(NullFormatter())
                ax.grid(alpha=0.3, which="both")
                if j == 0:
                    ax.set_ylabel(f"厚み {h} mm\n" + r"$\eta$ [Pa·s]")
                if i == 0:
                    ax.set_title(key, fontsize=10)
                if i == 3:
                    ax.set_xlabel(r"$\dot{\gamma}$ [1/s]")
        handles = [plt.Line2D([], [], color=c_, label=f"{t} ℃") for t, c_ in T_COL.items()]
        handles += [plt.Line2D([], [], color="gray", marker="o", ls="", label="学習の条件平均"), plt.Line2D([], [], color="gray", marker="s", mfc="none", ls="", label="学習に使わない条件")]
        axes[0, 0].legend(handles=handles, fontsize=7, loc="upper right")
        fig.suptitle(f"{MATS[mat]}・{ver} 版（{n} の学習）：設定温度ごとの予測曲線（線：MCMC の混合分布の平均、帯：95% 予測区間）", fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.98))
        save_fig(fig, OUT_DIR, f"fig7_curves_{ver}_{mat}")
        plt.show()

## 7b. 保存したファイル

In [ ]:
# セル14：保存したファイルの一覧
print("保存したファイル:")
for f in sorted(os.listdir(OUT_DIR)):
    if os.path.isfile(os.path.join(OUT_DIR, f)) and (not f.startswith("posterior_") or f == "posterior_summary.csv"):
        print("  ", os.path.normpath(os.path.join(OUT_DIR, f)))
print(f"  （事後サンプル posterior_<版>_<学習>_<材料>_<モデル>.csv：{len([f for f in os.listdir(OUT_DIR) if f.startswith('posterior_') and f != 'posterior_summary.csv'])} 個）")

## 8. 高解像度のグラフ（報告書・発表資料用）

せん断速度 ― 粘度の学習結果グラフを、GPR6月ノートのセル9-2 と同じ見せ方で、**600 dpi（15 × 10 インチ）**で 1 枚ずつ保存する。線は予測の平均、帯は 95% 区間、塗りつぶしの点は学習に使った条件、白抜きの点は学習に使わなかった条件（検証）、× は問題データ一覧の「感度確認」の条件。

- **モデル**：`HR_MODELS` に、MCMC で推定したモデル（`MCMC_MODELS` のもの。例：B_1、C_1）を並べる。曲線は、事後サンプルの予測分布の混合分布（平均と 95% 予測区間）。
- **レイアウト**：`"thickness"`＝厚みごとに 1 枚（3 つの設定温度の曲線）、`"temperature"`＝設定温度ごとに 1 枚（4 つの厚みの曲線。6月のノートと同じ）。
- **変えられるもの**（セルの先頭の設定エリア）：描くモデル・材料・学習の種類、帯（予測区間／信頼区間）、点（条件平均／ショットごと）、軸の範囲、予測線の延長、図の大きさ・解像度・文字・マーカーの大きさ、タイトルと軸名の表示、保存形式（png／pdf／svg）。
- 全体を一度実行して、学習が終わった後なら、このセルだけを何度でも実行し直せる。

In [ ]:
# セル15：高解像度のグラフ（せん断速度 ― 粘度。MCMC の混合分布の予測。GPR6月ノートのセル9-2 にならう）
# ==========================================
# ▼▼ 設定エリア（グラフの見た目・文字サイズ。このセルの設定だけを直して、このセルだけを実行し直せる） ▼▼
# ==========================================
HR_VERSION = DATA_VERSIONS[-1]     # どの版か："5data"（5data で学習。3data の条件は白抜き）／"Alldata"（全条件で学習）
HR_SET = None                      # 学習データ。None なら、版ごとの主な学習データ（5data 版：S_V、Alldata 版：ALL）
HR_MODELS = list(MCMC_MODELS)      # 描くモデル（MCMC_MODELS のもの）
HR_DRAWS = 60                      # 曲線に使う事後サンプルの数（増やすと遅い）
HR_MATERIALS = ["PP", "MIX"]       # 描く材料（PP：PP、MIX：粉砕材混合 PP）
HR_LAYOUTS = ["thickness", "temperature"]   # "thickness"：厚みごとに 1 枚（3 つの設定温度）／"temperature"：設定温度ごとに 1 枚（4 つの厚み。6月のノートと同じ）
HR_BAND = "predict"                # 帯："predict"＝95% 予測区間（ショット間のばらつき込み）／"latent"＝95% 信頼区間（平均の不確かさだけ）／None＝帯なし
HR_POINTS = "cond"                 # 点："cond"＝条件平均／"shot"＝ショットごと
HR_SHOW_TEST = True                # 学習に使わなかった条件（検証）を、白抜きで重ねるか
HR_MARK_SENS = True                # 問題データ一覧の「感度確認」の条件を、× で重ねるか
HR_X_RANGE = None                  # 予測線の範囲 [1/s]。None なら、各線のデータの範囲を HR_EXTEND 倍に延長。(30, 50000) のように指定すると、全部の線をその範囲に延長（外挿の確認用）
HR_EXTEND = 1.3
HR_XLIM = {"thickness": None, "temperature": (30, 50000)}   # 横軸（せん断速度 [1/s]）の表示範囲。(下限, 上限)＝固定（10 の累乗の目盛り。6月と同じ）／None＝その図のデータに合わせる（1・2・5 の目盛り）。
HR_YLIM = {"thickness": None, "temperature": (0.5, 5000)}   # 縦軸（粘度 [Pa·s]）の表示範囲。同じ書き方。レイアウトごとに変えられる（全部を同じ範囲にするなら、(30, 50000) のように書く）
HR_FIGSIZE = (15, 10)              # 図の大きさ（幅, 高さ）[インチ]
HR_DPI = 600                       # 解像度
HR_FONT = {"axis": 24, "legend": 24, "title": 24, "tick": 28}   # 軸名・凡例・タイトル・目盛り数字の文字サイズ
HR_MARKER = 200                    # 点の大きさ（条件平均のとき。面積）
HR_MARKER_SHOT = 70                # 点の大きさ（ショットごとのとき）
HR_LINE = 3.0                      # 予測線の太さ
HR_SHOW_TITLE = True               # タイトルを付けるか
HR_SHOW_XLABEL = False             # 軸名を付けるか（6月のノートと同じく、既定は付けない。資料に貼るとき、別に付ける場合のため）
HR_SHOW_YLABEL = False
HR_FORMATS = ["png"]               # 保存形式。"png"・"pdf"・"svg" から選ぶ（pdf・svg は、拡大しても粗くならない）
HR_SHOW = False                    # True なら、画面にも表示する（重い）
# ==========================================
HR_KW = dict(layouts=HR_LAYOUTS, band=HR_BAND, points=HR_POINTS, show_test=HR_SHOW_TEST, mark_sens=HR_MARK_SENS, x_range=HR_X_RANGE, extend=HR_EXTEND,
             xlim=HR_XLIM, ylim=HR_YLIM, figsize=HR_FIGSIZE, dpi=HR_DPI, font=HR_FONT, marker=HR_MARKER, marker_shot=HR_MARKER_SHOT, line=HR_LINE,
             show_title=HR_SHOW_TITLE, show_xlabel=HR_SHOW_XLABEL, show_ylabel=HR_SHOW_YLABEL, formats=HR_FORMATS, show=HR_SHOW)
HR_DIR = os.path.join(OUT_DIR, "highres")

hr_saved = []
for mat in HR_MATERIALS:
    n = HR_SET or main_set(HR_VERSION)
    pool = pool_of(mat, MAIN_DATA) if HR_VERSION == "5data" else D[(mat, "Alldata", MAIN_DATA)]
    tconds = set(TRAIN_DF[(mat, HR_VERSION, n)]["cond"]) if (mat, HR_VERSION, n) in TRAIN_DF else set()
    for key in HR_MODELS:
        c = (mat, HR_VERSION, n, key)
        if c not in traces:
            print(f"{MATS[mat]}・{HR_VERSION} 版・{n}・{key} は、サンプリングしていないので描かない（設定の DATA_VERSIONS・SETS5B・MCMC_MODELS に入れて、上から実行し直す）")
            continue
        t0 = time.time()
        st, hv = MODEL_SPEC[key]
        phis = thinned(c, HR_DRAWS)
        label = f"{key}：MCMC の混合分布（{HR_VERSION} 版・{n} の学習）"
        hr_saved += hr_flow(lambda Xg, c=c, st=st, hv=hv, phis=phis: tuple(predict_mixture(GD[c], st, hv, phis, Xg)[2:]), mat, MAIN_DATA, tconds, label, f"hr_{mat}_{HR_VERSION}_{n}_{key}_mcmc", HR_DIR, pool=pool, **HR_KW)
        print(f"{MATS[mat]} {key}：保存したファイル 累計 {len(hr_saved)} 個（{time.time() - t0:.0f} 秒）")
judge(len(hr_saved) > 0, f"高解像度のグラフを {len(hr_saved)} 個保存した（{os.path.normpath(HR_DIR)}）。形式 {HR_FORMATS}、{HR_DPI} dpi")